# Hiligaynon NER Fine-Tuning with mBERT and XLM-RoBERTa

This notebook demonstrates how to fine-tune multilingual transformer models (mBERT and XLM-RoBERTa) for Named Entity Recognition (NER) in Hiligaynon language.

## 1. Setup and Installation

In [ ]:
# Alternative: Install the latest compatible versions
!pip install --upgrade transformers
!pip install --upgrade accelerate
!pip install --upgrade datasets
!pip install seqeval evaluate

In [ ]:
import json
import torch
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load and Explore the Dataset

In [ ]:
# Load the Hiligaynon NER data
def load_hiligaynon_data(file_path):
    """Load Hiligaynon NER data from JSONL file"""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

# Upload your hiligaynon_ner_seed.jsonl file
from google.colab import files
uploaded = files.upload()

# Load the data
data = load_hiligaynon_data('final_combined_all.jsonl')
print(f"Total samples: {len(data)}")
print(f"Sample data: {data[0]}")

## 3. Data Analysis and Visualization

In [ ]:
# Analyze entity distribution
def analyze_entities(data):
    """Analyze entity types and their distribution"""
    entity_counts = Counter()
    entity_examples = {}

    for item in data:
        for start, end, label in item['entities']:
            entity_text = item['text'][start:end]
            entity_counts[label] += 1

            if label not in entity_examples:
                entity_examples[label] = []
            if len(entity_examples[label]) < 5:
                entity_examples[label].append(entity_text)

    return entity_counts, entity_examples

entity_counts, entity_examples = analyze_entities(data)

# Visualize entity distribution
plt.figure(figsize=(10, 6))
labels = list(entity_counts.keys())
counts = list(entity_counts.values())
plt.bar(labels, counts)
plt.xlabel('Entity Type')
plt.ylabel('Count')
plt.title('Distribution of Entity Types in Hiligaynon NER Dataset')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nEntity Distribution:")
for label, count in entity_counts.items():
    print(f"{label}: {count} entities")
    print(f"Examples: {', '.join(entity_examples[label][:3])}")
    print()

## 4. Data Preprocessing

In [ ]:
from transformers import AutoTokenizer
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class HiligaynonNERDataset(Dataset):
    """Custom dataset for Hiligaynon NER"""

    def __init__(self, texts, entities, tokenizer, max_length=128):
        self.texts = texts
        self.entities = entities
        self.tokenizer = tokenizer
        self.max_length = max_length

        # Define label mapping
        self.label_to_id = {
            'O': 0,
            'B-PER': 1,
            'I-PER': 2,
            'B-LOC': 3,
            'I-LOC': 4,
            'B-ORG': 5,
            'I-ORG': 6,
            'B-DATE': 7,
            'I-DATE': 8
        }
        self.id_to_label = {v: k for k, v in self.label_to_id.items()}

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        entities = self.entities[idx]

        # Tokenize text
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_offsets_mapping=True,
            return_tensors='pt'
        )

        # Create labels based on character offsets
        labels = self._create_labels(text, entities, encoding['offset_mapping'][0])

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': labels
        }

    def _create_labels(self, text, entities, offset_mapping):
        """Create token-level labels based on character-level annotations"""
        labels = torch.zeros(len(offset_mapping), dtype=torch.long)

        # Sort entities by start position
        sorted_entities = sorted(entities, key=lambda x: x[0])

        for start_char, end_char, entity_label in sorted_entities:
            # Entity label already contains B-/I- prefix, use it directly

            # Find tokens that overlap with entity
            for idx, (token_start, token_end) in enumerate(offset_mapping):
                if token_start == token_end:  # Special token
                    continue

                # Check if token overlaps with entity
                if token_start >= start_char and token_end <= end_char:
                    # For multi-token entities, we need to determine if this is B- or I-
                    if token_start == start_char:
                        # First token of entity - use the label as is (should be B-)
                        if entity_label in self.label_to_id:
                            labels[idx] = self.label_to_id[entity_label]
                        else:
                            print(f"Warning: Unknown label {entity_label}")
                    else:
                        # Subsequent tokens - convert B- to I-
                        if entity_label.startswith('B-'):
                            inside_label = 'I-' + entity_label[2:]
                            if inside_label in self.label_to_id:
                                labels[idx] = self.label_to_id[inside_label]
                            else:
                                print(f"Warning: Unknown label {inside_label}")
                        else:
                            # Use the label as is
                            if entity_label in self.label_to_id:
                                labels[idx] = self.label_to_id[entity_label]

        return labels

In [ ]:
# Prepare data for training
texts = [item['text'] for item in data]
entities = [item['entities'] for item in data]

# Split data into train, validation, and test sets
train_texts, test_texts, train_entities, test_entities = train_test_split(
    texts, entities, test_size=0.2, random_state=42
)
train_texts, val_texts, train_entities, val_entities = train_test_split(
    train_texts, train_entities, test_size=0.1, random_state=42
)

print(f"Train samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")
print(f"Test samples: {len(test_texts)}")

## 5. Model Configuration

In [ ]:
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset as HFDataset
import evaluate

# Choose model (mBERT or XLM-RoBERTa)
MODEL_CHOICE = "xlm-roberta"  # Change to "xlm-roberta" for XLM-RoBERTa

if MODEL_CHOICE == "mbert":
    model_name = "bert-base-multilingual-cased"
elif MODEL_CHOICE == "xlm-roberta":
    model_name = "xlm-roberta-base"
else:
    raise ValueError("Invalid model choice")

print(f"Using model: {model_name}")

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Create datasets
train_dataset = HiligaynonNERDataset(train_texts, train_entities, tokenizer)
val_dataset = HiligaynonNERDataset(val_texts, val_entities, tokenizer)
test_dataset = HiligaynonNERDataset(test_texts, test_entities, tokenizer)

# Initialize model
num_labels = len(train_dataset.label_to_id)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=train_dataset.id_to_label,
    label2id=train_dataset.label_to_id
)
model.to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 6. Define Evaluation Metrics

In [ ]:
# Load evaluation metric
seqeval_metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    """Compute NER metrics using seqeval"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    # Convert predictions and labels to list of labels
    true_labels = []
    pred_labels = []

    label_map = train_dataset.id_to_label

    for i in range(labels.shape[0]):
        true_label_seq = []
        pred_label_seq = []

        for j in range(labels.shape[1]):
            if labels[i, j] != -100:  # Ignore padding
                true_label_seq.append(label_map[labels[i, j]])
                pred_label_seq.append(label_map[predictions[i, j]])

        true_labels.append(true_label_seq)
        pred_labels.append(pred_label_seq)

    results = seqeval_metric.compute(
        predictions=pred_labels,
        references=true_labels,
        mode='strict',
        scheme='IOB2'
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

## 7. Training Configuration

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir=f"./hiligaynon-ner-{MODEL_CHOICE}",
    eval_strategy="epoch",  # Changed from evaluation_strategy
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    push_to_hub=False,
    report_to="none",
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),
)

# Data collator
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if training_args.fp16 else None,
    padding=True,
    label_pad_token_id=-100
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

## 8. Train the Model

In [ ]:
# Train the model
print("Starting training...")
train_results = trainer.train()

# Save the model
trainer.save_model(f"./hiligaynon-ner-{MODEL_CHOICE}-final")
tokenizer.save_pretrained(f"./hiligaynon-ner-{MODEL_CHOICE}-final")

In [ ]:
# Plot training history
if hasattr(trainer.state, 'log_history'):
    history = pd.DataFrame(trainer.state.log_history)

    # Plot loss
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    train_loss = history[history['loss'].notna()]['loss']
    plt.plot(train_loss, label='Training Loss')
    if 'eval_loss' in history:
        eval_loss = history[history['eval_loss'].notna()]['eval_loss']
        plt.plot(eval_loss.index, eval_loss, label='Validation Loss')
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()

    # Plot F1 score
    plt.subplot(1, 2, 2)
    if 'eval_f1' in history:
        eval_f1 = history[history['eval_f1'].notna()]['eval_f1']
        plt.plot(eval_f1.index, eval_f1, label='Validation F1')
        plt.xlabel('Steps')
        plt.ylabel('F1 Score')
        plt.title('Validation F1 Score')
        plt.legend()

    plt.tight_layout()
    plt.show()

## 9. Evaluate on Test Set

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
test_results = trainer.evaluate(eval_dataset=test_dataset)

print("\nTest Results:")
for metric, value in test_results.items():
    if metric.startswith('eval_'):
        print(f"{metric[5:]}: {value:.4f}")

## 10. Inference and Visualization

In [ ]:
def predict_entities(text, model, tokenizer, device):
    """Predict entities for a given text"""
    # Tokenize
    inputs = tokenizer(
        text,
        truncation=True,
        padding=True,
        return_tensors="pt",
        return_offsets_mapping=True
    )

    offset_mapping = inputs.pop('offset_mapping')[0]
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Get predictions
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)[0]

    # Convert predictions to entities
    entities = []
    current_entity = None

    for idx, (pred_id, (start, end)) in enumerate(zip(predictions, offset_mapping)):
        if start == end:  # Special token
            continue

        label = train_dataset.id_to_label[pred_id.item()]

        if label.startswith('B-'):
            # Save previous entity if exists
            if current_entity:
                entities.append(current_entity)

            # Start new entity
            current_entity = {
                'text': text[start:end],
                'label': label[2:],
                'start': start.item(),
                'end': end.item()
            }
        elif label.startswith('I-') and current_entity and current_entity['label'] == label[2:]:
            # Continue current entity
            current_entity['text'] = text[current_entity['start']:end]
            current_entity['end'] = end.item()
        else:
            # End current entity
            if current_entity:
                entities.append(current_entity)
                current_entity = None

    # Don't forget the last entity
    if current_entity:
        entities.append(current_entity)

    return entities

In [ ]:
# Test on sample sentences
test_sentences = [
    "Backrider sang Criminology graduate nga napatay sa aksidente sa Manapla, nagtaliwan man.",
    "Sitwasyon sa Rizal Elementary School sa Bacolod City subong nga adlaw, Hunyo 19 , kadungan sang pagdumdom sang 164th nga kaadlawan ni Dr. Jose P Rizal.",
    "Mga pag-atake sang Iran sa Israel, nagbaskog pa",
    "Pagpadala sang US military assets sa Israel para sa pag-atake sa Iran, naghagan-hagan",
]

for sentence in test_sentences:
    print(f"\nText: {sentence}")
    entities = predict_entities(sentence, model, tokenizer, device)
    print("Detected entities:")
    for entity in entities:
        print(f"  - {entity['text']} ({entity['label']})")

## 11. Error Analysis

In [ ]:
def perform_error_analysis(model, dataset, tokenizer, device, num_samples=20):
    """Analyze prediction errors"""
    errors = []
    correct = []

    for i in range(min(num_samples, len(dataset))):
        item = dataset[i]
        text = dataset.texts[i]
        true_entities = dataset.entities[i]

        # Get predictions
        predicted_entities = predict_entities(text, model, tokenizer, device)

        # Convert true entities to comparable format
        true_entity_set = {(e[0], e[1], e[2]) for e in true_entities}
        pred_entity_set = {(e['start'], e['end'], e['label']) for e in predicted_entities}

        # Find errors
        false_negatives = true_entity_set - pred_entity_set
        false_positives = pred_entity_set - true_entity_set

        if false_negatives or false_positives:
            errors.append({
                'text': text,
                'true_entities': true_entities,
                'predicted_entities': predicted_entities,
                'false_negatives': list(false_negatives),
                'false_positives': list(false_positives)
            })
        else:
            correct.append(text)

    return errors, correct

# Perform error analysis
errors, correct = perform_error_analysis(model, test_dataset, tokenizer, device)

print(f"Correct predictions: {len(correct)}/{len(correct) + len(errors)}")
print(f"\nSample errors:")
for i, error in enumerate(errors[:3]):
    print(f"\n{i+1}. Text: {error['text']}")
    print(f"   False negatives: {error['false_negatives']}")
    print(f"   False positives: {error['false_positives']}")

## 12. Model Comparison (mBERT vs XLM-RoBERTa)

In [ ]:
# To compare models, run this notebook twice with different MODEL_CHOICE values
# and save the results

model_results = {
    'Model': MODEL_CHOICE,
    'Precision': test_results.get('eval_precision', 0),
    'Recall': test_results.get('eval_recall', 0),
    'F1': test_results.get('eval_f1', 0),
    'Accuracy': test_results.get('eval_accuracy', 0)
}

# Save results
with open(f'{MODEL_CHOICE}_results.json', 'w') as f:
    json.dump(model_results, f)

print(f"\nResults saved for {MODEL_CHOICE}")

## 13. Save Model for Deployment

In [ ]:
# Save the model in a format ready for deployment
model.save_pretrained(f"hiligaynon-ner-{MODEL_CHOICE}-deploy")
tokenizer.save_pretrained(f"hiligaynon-ner-{MODEL_CHOICE}-deploy")

# Create a config file with label mappings
config = {
    'model_name': model_name,
    'label_to_id': train_dataset.label_to_id,
    'id_to_label': train_dataset.id_to_label,
    'max_length': 128
}

with open(f"hiligaynon-ner-{MODEL_CHOICE}-deploy/ner_config.json", 'w') as f:
    json.dump(config, f, indent=2)

print(f"Model saved to hiligaynon-ner-{MODEL_CHOICE}-deploy/")

# Download the model
from google.colab import files
!zip -r hiligaynon-ner-{MODEL_CHOICE}-deploy.zip hiligaynon-ner-{MODEL_CHOICE}-deploy/
files.download(f'hiligaynon-ner-{MODEL_CHOICE}-deploy.zip')

## 14. Inference

Inference with BIO tags

In [ ]:
# Cell 1: Add BIO-preserving prediction and analysis functions
def predict_entities_with_bio(text, model, tokenizer, device):
    """
    Predict entities for a given text, preserving B-/I- tags
    """
    # Tokenize
    inputs = tokenizer(
        text,
        truncation=True,
        padding=True,
        return_tensors="pt",
        return_offsets_mapping=True
    )

    offset_mapping = inputs.pop('offset_mapping')[0]
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Get predictions
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)[0]

    # Convert predictions to entities with BIO tags preserved
    entities = []

    for idx, (pred_id, (start, end)) in enumerate(zip(predictions, offset_mapping)):
        if start == end:  # Special token
            continue

        label = train_dataset.id_to_label[pred_id.item()]

        if label != 'O':  # If it's an entity
            entities.append({
                'text': text[start:end],
                'label': label,  # Keep the full label with B-/I- prefix
                'label_type': label[2:] if label.startswith(('B-', 'I-')) else label,  # Extract type
                'start': start.item(),
                'end': end.item()
            })

    return entities

def display_entities_with_bio(text, entities):
    """
    Display entities with both BIO tags and entity types
    """
    print(f"\nText: {text}")
    print("-" * 80)

    if entities:
        print("Detected Entities:")
        for i, entity in enumerate(entities, 1):
            bio_tag = entity['label']
            entity_type = entity['label_type']
            print(f"  {i}. {entity['text']} ({bio_tag} / Type: {entity_type}) - Position: [{entity['start']}, {entity['end']}]")
    else:
        print("No entities detected.")
    print()

Test BIO tags

In [ ]:
# Cell 2: Test inference with BIO tags preserved
print("=" * 80)
print("INFERENCE WITH BIO TAGS PRESERVED")
print("=" * 80)

test_sentences = [
    "Si Maria kag si Juan nagkadto sa Bacolod.",
    "Ang Presidente Marcos naghambal sa UN Assembly.",
    "Nagtalipon ang mga tawo sa Manila",
    "Bangkay sang lalaki, nasapwan nga ginlubong sa kampo sa Murcia.",
]

for text in test_sentences:
    entities = predict_entities_with_bio(text, model, tokenizer, device)
    display_entities_with_bio(text, entities)

## Error analysis

In [ ]:
# Cell 3: Enhanced error analysis with BIO tags
def perform_error_analysis_with_bio(model, dataset, tokenizer, device, num_samples=20):
    """
    Analyze prediction errors while preserving B-/I- tags
    """
    errors = []
    correct = []

    for i in range(min(num_samples, len(dataset))):
        text = dataset.texts[i]
        true_entities = dataset.entities[i]

        # Get predictions
        predicted_entities = predict_entities_with_bio(text, model, tokenizer, device)

        # Convert to comparable format (start, end, full_label)
        true_entity_set = {(e[0], e[1], e[2]) for e in true_entities}
        pred_entity_set = {(e['start'], e['end'], e['label']) for e in predicted_entities}

        # Find errors
        false_negatives = true_entity_set - pred_entity_set
        false_positives = pred_entity_set - true_entity_set

        if false_negatives or false_positives:
            errors.append({
                'text': text,
                'true_entities': true_entities,
                'predicted_entities': predicted_entities,
                'false_negatives': list(false_negatives),
                'false_positives': list(false_positives)
            })
        else:
            correct.append(text)

    return errors, correct

# Run the analysis
errors, correct = perform_error_analysis_with_bio(model, test_dataset, tokenizer, device, num_samples=20)

print(f"Correct predictions: {len(correct)}/{len(correct) + len(errors)}")
print(f"\nSample errors with BIO tags:")
for i, error in enumerate(errors[:3]):  # Show first 3 errors
    print(f"\n{i+1}. Text: {error['text']}")
    print(f"   False negatives (missing): {error['false_negatives']}")
    print(f"   False positives (incorrect): {error['false_positives']}")

Random Error Analysis

In [ ]:
import random          # ⬅️  NEW: for random sampling

# ---------------------------------------------------------------------
# Modified: now chooses `num_samples` random, non-repeating indices
# ---------------------------------------------------------------------
def perform_error_analysis_with_bio(model, dataset, tokenizer, device,
                                    num_samples=20, seed: int | None = None):
    """
    Analyze prediction errors on a random subset (up to `num_samples`)
    while preserving B-/I- tags.
    """
    # Optional seed for reproducible runs
    if seed is not None:
        random.seed(seed)

    # 1️⃣ Pick distinct random indices (caps at dataset length)
    sample_idxs = random.sample(range(len(dataset)),
                                k=min(num_samples, len(dataset)))

    errors, correct = [], []

    # 2️⃣ Loop over those random indices instead of the first `num_samples`
    for idx in sample_idxs:
        text = dataset.texts[idx]
        true_entities = dataset.entities[idx]

        # Get predictions
        predicted_entities = predict_entities_with_bio(text, model, tokenizer, device)

        # Convert to comparable format (start, end, full_label)
        true_entity_set = {(e[0], e[1], e[2]) for e in true_entities}
        pred_entity_set = {(e['start'], e['end'], e['label']) for e in predicted_entities}

        # Find errors
        false_negatives = true_entity_set - pred_entity_set
        false_positives = pred_entity_set - true_entity_set

        if false_negatives or false_positives:
            errors.append({
                'text': text,
                'true_entities': true_entities,
                'predicted_entities': predicted_entities,
                'false_negatives': list(false_negatives),
                'false_positives': list(false_positives)
            })
        else:
            correct.append(text)

    return errors, correct

    # Call the function
    errors, correct = perform_error_analysis_with_bio(
    model,
    test_dataset,
    tokenizer,
    device,
    num_samples=50,   # or any number you like
    seed=42           # omit for fresh randomness each run
    )

    # Basic summary
    print(f"Perfect predictions: {len(correct)}/{len(errors) + len(correct)}")

    # Peek at the first few errors
    for i, err in enumerate(errors[:3], 1):
        print(f"\n{i}. Text: {err['text']}")
        print("   False negatives:", err['false_negatives'])
        print("   False positives:", err['false_positives'])



In [ ]:
# 🟢 CELL — define prediction helper, run error analysis, print results
# --------------------------------------------------------------------
import torch, random

def predict_entities_with_bio(text, model, tokenizer, device):
    """
    Return BIO-tagged entities with char-level offsets.
    Output: [{"start": s, "end": e, "label": "B-PER"}, …]
    """
    # 1️⃣ Tokenize (keep offsets) -------------------------------------
    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        return_tensors="pt"
    )

    offset_mapping = enc.pop("offset_mapping")[0].tolist()   # remove so model won’t complain
    enc = {k: v.to(device) for k, v in enc.items()}

    # 2️⃣ Forward pass -----------------------------------------------
    with torch.no_grad():
        logits = model(**enc).logits

    preds      = torch.argmax(logits, dim=-1)[0].tolist()
    id2label   = model.config.id2label
    entities   = []

    # 3️⃣ Convert token-level BIO predictions → char spans -----------
    for (start, end), pred_id in zip(offset_mapping, preds):
        if start == end:                 # special tokens -> (0,0)
            continue
        label = id2label[pred_id]
        if label == "O":
            continue
        entities.append({"start": start, "end": end, "label": label})

    return entities


# --------------------------------------------------------------------
# 🔍  Run the random-subset error analysis & print a quick report
# --------------------------------------------------------------------
NUM_SAMPLES = 50      # how many random test sentences to inspect
SEED        = 42      # set to None for fresh randomness each run
SHOW_N      = 5       # how many error cases to print in full

errors, correct = perform_error_analysis_with_bio(
    model,
    test_dataset,      # <-- your Dataset object
    tokenizer,
    device,
    num_samples = NUM_SAMPLES,
    seed        = SEED
)

print(f"\n── Error analysis on {NUM_SAMPLES} random sentences ──")
print(f"Perfect predictions : {len(correct)}/{NUM_SAMPLES}")
print(f"Sentences with errors: {len(errors)}\n")

for i, err in enumerate(errors[:SHOW_N], 1):
    print(f"{i}. Text: {err['text']}")
    print("   False negatives:", err['false_negatives'])
    print("   False positives :", err['false_positives'], "\n")


Visualize BIO tags

In [ ]:
# Cell 4: Visualize BIO tagging
def visualize_bio_tags(text, entities):
    """
    Create a visual representation of BIO tags
    """
    print(f"\nText: {text}")
    print("Entities:")
    for e in entities:
        print(f"  {e['text']}: {e['label']} at [{e['start']}-{e['end']}]")

# Test on a sample
sample_text = "Si Maria kag si Juan nagkadto sa Manila."
entities = predict_entities_with_bio(sample_text, model, tokenizer, device)
visualize_bio_tags(sample_text, entities)

Predict Entities

In [ ]:
def predict_entities_with_bio_merged(text, model, tokenizer, device):
    """
    Predict entities and merge subword tokens properly
    """
    # Tokenize
    inputs = tokenizer(
        text,
        truncation=True,
        padding=True,
        return_tensors="pt",
        return_offsets_mapping=True
    )

    offset_mapping = inputs.pop('offset_mapping')[0]
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Get predictions
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)[0]

    # Get tokens for debugging
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    # Process predictions and merge subwords
    entities = []
    current_entity = None

    for idx, (pred_id, (start, end), token) in enumerate(zip(predictions, offset_mapping, tokens)):
        if start == end:  # Special token
            continue

        label = train_dataset.id_to_label[pred_id.item()]

        if label == 'O':
            # Not an entity, save current if exists
            if current_entity:
                entities.append(current_entity)
                current_entity = None
        else:
            # This is an entity token
            if label.startswith('B-'):
                # Beginning of new entity
                if current_entity:
                    entities.append(current_entity)

                current_entity = {
                    'text': text[start:end],
                    'label': label,
                    'label_type': label[2:],
                    'start': start.item(),
                    'end': end.item(),
                    'tokens': [token]
                }
            elif label.startswith('I-') and current_entity:
                # Continuation of current entity
                if current_entity['label_type'] == label[2:]:
                    # Same entity type, extend it
                    current_entity['end'] = end.item()
                    current_entity['text'] = text[current_entity['start']:current_entity['end']]
                    current_entity['tokens'].append(token)
                    # Update label to show it's continued
                    if not current_entity['label'].startswith('B-'):
                        current_entity['label'] = label
                else:
                    # Different entity type, save current and start new
                    entities.append(current_entity)
                    current_entity = {
                        'text': text[start:end],
                        'label': label,
                        'label_type': label[2:],
                        'start': start.item(),
                        'end': end.item(),
                        'tokens': [token]
                    }
            else:
                # Shouldn't happen with proper BIO tagging
                if current_entity:
                    entities.append(current_entity)
                current_entity = None

    # Don't forget the last entity
    if current_entity:
        entities.append(current_entity)

    return entities

def display_entities_with_tokens(text, entities):
    """
    Display entities with token information
    """
    print(f"\nText: {text}")
    print("-" * 80)

    if entities:
        print("Detected Entities:")
        for i, entity in enumerate(entities, 1):
            bio_tag = entity['label']
            entity_type = entity['label_type']
            tokens_str = " ".join(entity['tokens'])
            print(f"  {i}. '{entity['text']}' ({bio_tag} → {entity_type})")
            print(f"     Position: [{entity['start']}, {entity['end']}]")
            print(f"     Tokens: {tokens_str}")
    else:
        print("No entities detected.")
    print()

# Test with the problematic sentence
test_text = "Backrider sang Criminology graduate nga napatay sa aksidente sa Manapla, nagtaliwan man."
entities = predict_entities_with_bio_merged(test_text, model, tokenizer, device)
display_entities_with_tokens(test_text, entities)

# Also test other sentences
print("\n" + "=" * 80)
print("TESTING OTHER SENTENCES")
print("=" * 80)

test_sentences = [
    "Si Maria kag si Juan nagkadto sa Manila.",
    "Ang University of the Philippines sa Diliman.",
    "Si Dr. Jose Rizal ang pambansang bayani.",
]

for text in test_sentences:
    entities = predict_entities_with_bio_merged(text, model, tokenizer, device)
    display_entities_with_tokens(text, entities)

## k-fold Cross Validation

Check datasets

In [ ]:
[k for k,v in globals().items() if hasattr(v, "__len__") and not k.startswith("_")]


mbert k-fold

In [ ]:
# ============================================
# 🔵 mBERT 5-Fold Cross-Validation
# ============================================

CHECKPOINT = "/content/hiligaynon-ner-mbert-deploy"
print(f"📂  mBERT checkpoint: {CHECKPOINT}")

from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          DataCollatorForTokenClassification,
                          Trainer, TrainingArguments)
from torch.utils.data import ConcatDataset, Subset
from sklearn.model_selection import KFold
import torch, numpy as np, pandas as pd
from tabulate import tabulate

# 1️⃣  Tokeniser & datasets
tokenizer_mbert = AutoTokenizer.from_pretrained(CHECKPOINT)

train_dataset_mbert = HiligaynonNERDataset(
    train_texts, train_entities, tokenizer_mbert
)
val_dataset_mbert = HiligaynonNERDataset(
    val_texts,   val_entities,   tokenizer_mbert
)

data_collator_mbert = DataCollatorForTokenClassification(
    tokenizer_mbert, label_pad_token_id=-100
)

# 2️⃣  CV loop
full_ds = ConcatDataset([train_dataset_mbert, val_dataset_mbert])
kf      = KFold(n_splits=5, shuffle=True, random_state=42)
metrics_mbert = []

def model_init_mbert():
    return AutoModelForTokenClassification.from_pretrained(CHECKPOINT)


for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(len(full_ds))), 1):
    print(f"\n—— mBERT Fold {fold}/5 ——————————————")
    tr_ds, va_ds = Subset(full_ds, tr_idx), Subset(full_ds, va_idx)

    args = TrainingArguments(
        output_dir=f"./results/mbert_fold{fold}",
        eval_strategy="epoch",
        save_strategy="no",
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=5e-5,
        weight_decay=0.01,
        seed=42,
        report_to="none",
        logging_steps=50,
    )

    trainer = Trainer(
        model_init=model_init_mbert,
        args=args,
        train_dataset=tr_ds,
        eval_dataset=va_ds,
        tokenizer=tokenizer_mbert,
        data_collator=data_collator_mbert,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    res = trainer.evaluate()
    metrics_mbert.append(res)
    print(f"Fold {fold} F1: {res['eval_f1']:.4f}")

# 3️⃣  Report
df_mbert = pd.DataFrame([{
        "Precision": m["eval_precision"],
        "Recall":    m["eval_recall"],
        "F1":        m["eval_f1"],
        "Accuracy":  m["eval_accuracy"],
    } for m in metrics_mbert],
    index=[f"Fold {i+1}" for i in range(5)]
)
summary = df_mbert.agg(["mean", "std"]).rename(index={"mean": "Mean", "std": "Std"})
print("\n### mBERT K-Fold Metrics")
print(tabulate(pd.concat([df_mbert, summary]), headers="keys",
               tablefmt="github", floatfmt=".4f"))


roberta k-fold

In [ ]:
# ============================================
# 🟣 XLM-R 5-Fold Cross-Validation
# ============================================

CHECKPOINT = "/content/hiligaynon-ner-xlm-roberta-deploy"
print(f"📂  XLM-R checkpoint: {CHECKPOINT}")

from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          DataCollatorForTokenClassification,
                          Trainer, TrainingArguments)
from torch.utils.data import ConcatDataset, Subset
from sklearn.model_selection import KFold
import torch, numpy as np, pandas as pd
from tabulate import tabulate

# 1️⃣  Tokeniser & datasets
tokenizer_xlmr = AutoTokenizer.from_pretrained(CHECKPOINT)

train_dataset_xlmr = HiligaynonNERDataset(
    train_texts, train_entities, tokenizer_xlmr
)
val_dataset_xlmr = HiligaynonNERDataset(
    val_texts,   val_entities,   tokenizer_xlmr
)

data_collator_xlmr = DataCollatorForTokenClassification(
    tokenizer_xlmr, label_pad_token_id=-100
)

# 2️⃣  CV loop
full_ds = ConcatDataset([train_dataset_xlmr, val_dataset_xlmr])
kf      = KFold(n_splits=5, shuffle=True, random_state=42)
metrics_xlmr = []

def model_init_xlmr():
    return AutoModelForTokenClassification.from_pretrained(CHECKPOINT)


for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(len(full_ds))), 1):
    print(f"\n—— XLM-R Fold {fold}/5 ——————————————")
    tr_ds, va_ds = Subset(full_ds, tr_idx), Subset(full_ds, va_idx)

    args = TrainingArguments(
        output_dir=f"./results/xlmr_fold{fold}",
        eval_strategy="epoch",
        save_strategy="no",
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=5e-5,
        weight_decay=0.01,
        seed=42,
        report_to="none",
        logging_steps=50,
    )

    trainer = Trainer(
        model_init=model_init_xlmr,
        args=args,
        train_dataset=tr_ds,
        eval_dataset=va_ds,
        tokenizer=tokenizer_xlmr,
        data_collator=data_collator_xlmr,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    res = trainer.evaluate()
    metrics_xlmr.append(res)
    print(f"Fold {fold} F1: {res['eval_f1']:.4f}")

# 3️⃣  Report
df_xlmr = pd.DataFrame([{
        "Precision": m["eval_precision"],
        "Recall":    m["eval_recall"],
        "F1":        m["eval_f1"],
        "Accuracy":  m["eval_accuracy"],
    } for m in metrics_xlmr],
    index=[f"Fold {i+1}" for i in range(5)]
)
summary = df_xlmr.agg(["mean", "std"]).rename(index={"mean": "Mean", "std": "Std"})
print("\n### XLM-R K-Fold Metrics")
print(tabulate(pd.concat([df_xlmr, summary]), headers="keys",
               tablefmt="github", floatfmt=".4f"))


In [ ]:
print(tokenizer.name_or_path)          # → check which tokenizer the dataset used
print(model.config.vocab_size)         # → 119547 for mBERT
print(int(train_dataset[0]["input_ids"][0]))  # a few ids; if any > vocab_size → mismatch


## mBERT Confusion Matrix

In [ ]:
# ============================================
# 🔵 FIXED mBERT Confusion Matrix
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from collections import defaultdict

def generate_confusion_matrix_mbert(model, dataset, tokenizer, device, model_name="mBERT"):
    """
    Generate confusion matrix for mBERT model - handles missing labels in test set
    """
    print(f"Generating confusion matrix for {model_name}...")

    all_true_labels = []
    all_pred_labels = []

    # Get predictions for entire dataset
    for i in range(len(dataset)):
        item = dataset[i]
        text = dataset.texts[i]

        # Get model predictions
        inputs = tokenizer(
            text,
            truncation=True,
            padding=True,
            return_tensors="pt",
            return_offsets_mapping=True
        )

        offset_mapping = inputs.pop('offset_mapping')[0]
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.argmax(outputs.logits, dim=-1)[0]

        # Get true labels
        true_labels = item['labels']

        # Align predictions with true labels
        for j, (pred_id, true_id, (start, end)) in enumerate(zip(predictions, true_labels, offset_mapping)):
            if start == end:  # Skip special tokens
                continue
            if true_id == -100:  # Skip padding
                continue

            all_true_labels.append(true_id.item() if hasattr(true_id, 'item') else true_id)
            all_pred_labels.append(pred_id.item())

    # Find unique labels that actually appear in the data
    unique_labels = sorted(list(set(all_true_labels + all_pred_labels)))
    print(f"Labels present in test set: {unique_labels}")
    print(f"Missing labels: {[i for i in range(len(dataset.id_to_label)) if i not in unique_labels]}")

    # Create label names only for labels that exist
    label_names = [dataset.id_to_label[i] for i in unique_labels]
    print(f"Label names in confusion matrix: {label_names}")

    # Generate confusion matrix with explicit label ordering
    cm = confusion_matrix(all_true_labels, all_pred_labels, labels=unique_labels)

    # Create figure
    plt.figure(figsize=(10, 8))

    # Plot confusion matrix
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_names,
                yticklabels=label_names,
                cbar_kws={'label': 'Count'})

    plt.title(f'Confusion Matrix - {model_name} on Hiligaynon NER\n(Note: B-DATE and I-DATE not present in test set)', fontsize=14)
    plt.xlabel('Predicted Labels', fontsize=12)
    plt.ylabel('True Labels', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # Calculate per-class metrics with explicit labels
    print(f"\n{model_name} Per-Class Metrics:")
    print("="*60)

    report = classification_report(
        all_true_labels,
        all_pred_labels,
        labels=unique_labels,  # Only use labels that exist
        target_names=label_names,  # Only names for existing labels
        zero_division=0
    )
    print(report)

    # Additional analysis
    print("\nEntity Type Summary:")
    print("-"*40)
    entity_types = {'PER': 0, 'LOC': 0, 'ORG': 0, 'DATE': 0}
    for label_id in all_true_labels:
        label_name = dataset.id_to_label[label_id]
        if label_name != 'O':
            entity_type = label_name.split('-')[1]
            entity_types[entity_type] += 1

    for entity_type, count in entity_types.items():
        print(f"{entity_type}: {count} tokens")

    return cm, all_true_labels, all_pred_labels

# Load the trained mBERT model
mbert_model_path = f"./hiligaynon-ner-mbert-final"
mbert_model = AutoModelForTokenClassification.from_pretrained(mbert_model_path)
mbert_model.to(device)
mbert_model.eval()

# Generate confusion matrix for test dataset
cm_mbert, true_labels_mbert, pred_labels_mbert = generate_confusion_matrix_mbert(
    mbert_model, test_dataset, tokenizer, device, "mBERT"
)

Version 2

In [ ]:
# ============================================
# 🔵 mBERT Confusion Matrix  (no "O")
# ============================================

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from transformers import AutoModelForTokenClassification

def generate_confusion_matrix_mbert(model,
                                    dataset,
                                    tokenizer,
                                    device,
                                    model_name="mBERT"):
    """
    Build a confusion matrix & per-class report for `model`,
    **excluding the 'O' tag**.
    """
    print(f"Generating confusion matrix for {model_name} (excluding 'O')...")

    id2label = model.config.id2label          # use model’s mapping
    label2id = {v: k for k, v in id2label.items()}
    o_id     = label2id["O"]                  # numeric ID for "O"

    all_true, all_pred = [], []

    # ── iterate over the whole dataset ────────────────────────────────
    for idx in range(len(dataset)):
        item = dataset[idx]
        text = item["text"] if "text" in item else dataset.texts[idx]

        # True token-level IDs
        true_labels = item["labels"]

        # Encode text
        inputs = tokenizer(
            text,
            truncation=True,
            padding=True,
            return_tensors="pt",
            return_offsets_mapping=True
        )
        offsets = inputs.pop("offset_mapping")[0]
        inputs  = {k: v.to(device) for k, v in inputs.items()}

        # Forward pass
        with torch.no_grad():
            preds = torch.argmax(model(**inputs).logits, dim=-1)[0]

        # Align token by token
        for pred_id, true_id, (start, end) in zip(preds, true_labels, offsets):
            if start == end or true_id == -100:     # special / padding
                continue

            true_id = int(true_id)    # ensure Python int
            pred_id = int(pred_id)

            # If **both** are 'O', ignore this position completely
            if true_id == o_id and pred_id == o_id:
                continue

            all_true.append(true_id)
            all_pred.append(pred_id)

    # ── keep only labels that appear and are not 'O' ─────────────────
    unique_ids   = sorted({lab for lab in all_true + all_pred if lab != o_id})
    label_names  = [id2label[i] for i in unique_ids]

    print(f"Labels in confusion matrix: {label_names}")

    # ── confusion matrix ─────────────────────────────────────────────
    cm = confusion_matrix(all_true, all_pred, labels=unique_ids)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm,
                annot=True,
                fmt="d",
                cmap="Blues",
                xticklabels=label_names,
                yticklabels=label_names,
                cbar_kws={"label": "Count"})
    plt.title(f"Confusion Matrix – {model_name} (no 'O')", fontsize=14)
    plt.xlabel("Predicted", fontsize=12)
    plt.ylabel("True", fontsize=12)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # ── per-class metrics ───────────────────────────────────────────
    print(f"\n{model_name} Per-Class Metrics (excluding 'O')\n" + "="*60)
    print(
        classification_report(all_true,
                              all_pred,
                              labels=unique_ids,
                              target_names=label_names,
                              zero_division=0)
    )

    return cm, all_true, all_pred


# --------------------------------------------------------------------
# Load the trained mBERT model
mbert_model_path = "./hiligaynon-ner-mbert-final"
mbert_model = (AutoModelForTokenClassification
               .from_pretrained(mbert_model_path)
               .to(device))
mbert_model.eval()

# Generate confusion matrix
cm_mbert, true_labels_mbert, pred_labels_mbert = generate_confusion_matrix_mbert(
    mbert_model,
    test_dataset,
    tokenizer,
    device,
    model_name="mBERT"
)


In [ ]:
print("id_to_label mapping:")
print(test_dataset.id_to_label)
print(f"Total labels: {len(test_dataset.id_to_label)}")

In [ ]:
# Check unique labels in test dataset
unique_labels = set()
for i in range(len(test_dataset)):
    item = test_dataset[i]
    labels = item['labels']
    for label in labels:
        if label != -100:  # Skip padding
            unique_labels.add(label.item() if hasattr(label, 'item') else label)

print(f"Unique labels in test set: {sorted(unique_labels)}")
print(f"Number of unique labels: {len(unique_labels)}")

## XLM-RoBERTa Confusion Matrix

In [ ]:
# ============================================
# 🟣 FIXED XLM-RoBERTa Confusion Matrix
# ============================================

def generate_confusion_matrix_xlmr(model, dataset, tokenizer, device, model_name="XLM-RoBERTa"):
    """
    Generate confusion matrix for XLM-RoBERTa model - handles missing labels in test set
    """
    print(f"\nGenerating confusion matrix for {model_name}...")

    all_true_labels = []
    all_pred_labels = []

    # Get predictions for entire dataset
    for i in range(len(dataset)):
        item = dataset[i]
        text = dataset.texts[i]

        # Get model predictions
        inputs = tokenizer(
            text,
            truncation=True,
            padding=True,
            return_tensors="pt",
            return_offsets_mapping=True
        )

        offset_mapping = inputs.pop('offset_mapping')[0]
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.argmax(outputs.logits, dim=-1)[0]

        # Get true labels
        true_labels = item['labels']

        # Align predictions with true labels
        for j, (pred_id, true_id, (start, end)) in enumerate(zip(predictions, true_labels, offset_mapping)):
            if start == end:  # Skip special tokens
                continue
            if true_id == -100:  # Skip padding
                continue

            all_true_labels.append(true_id.item() if hasattr(true_id, 'item') else true_id)
            all_pred_labels.append(pred_id.item())

    # Find unique labels that actually appear in the data
    unique_labels = sorted(list(set(all_true_labels + all_pred_labels)))

    # Create label names only for labels that exist
    label_names = [dataset.id_to_label[i] for i in unique_labels]

    # Generate confusion matrix
    cm = confusion_matrix(all_true_labels, all_pred_labels, labels=unique_labels)

    # Create figure
    plt.figure(figsize=(10, 8))

    # Plot confusion matrix with purple colormap for XLM-R
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
                xticklabels=label_names,
                yticklabels=label_names,
                cbar_kws={'label': 'Count'})

    plt.title(f'Confusion Matrix - {model_name} on Hiligaynon NER\n(Note: B-DATE and I-DATE not present in test set)', fontsize=14)
    plt.xlabel('Predicted Labels', fontsize=12)
    plt.ylabel('True Labels', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # Calculate per-class metrics
    print(f"\n{model_name} Per-Class Metrics:")
    print("="*60)
    report = classification_report(
        all_true_labels,
        all_pred_labels,
        labels=unique_labels,
        target_names=label_names,
        zero_division=0
    )
    print(report)

    return cm, all_true_labels, all_pred_labels

# Load the trained XLM-RoBERTa model
xlmr_model_path = f"./hiligaynon-ner-xlm-roberta-final"
xlmr_model = AutoModelForTokenClassification.from_pretrained(xlmr_model_path)
xlmr_model.to(device)
xlmr_model.eval()

# Note: You need to create a new test dataset with XLM-RoBERTa tokenizer
xlmr_tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
test_dataset_xlmr = HiligaynonNERDataset(test_texts, test_entities, xlmr_tokenizer)

# Generate confusion matrix for test dataset
cm_xlmr, true_labels_xlmr, pred_labels_xlmr = generate_confusion_matrix_xlmr(
    xlmr_model, test_dataset_xlmr, xlmr_tokenizer, device, "XLM-RoBERTa"
)


In [ ]:
Version 2

In [ ]:
# ============================================
# 🟣 XLM-R Confusion Matrix  (-- no “O” tag)
# ============================================
#  – Stand-alone: just run after `device`, `test_texts`,
#    `test_entities`, and `HiligaynonNERDataset` are defined.

from transformers import AutoTokenizer, AutoModelForTokenClassification
from sklearn.metrics import confusion_matrix, classification_report
import torch, numpy as np, matplotlib.pyplot as plt, seaborn as sns

def confusion_matrix_xlmr_no_O(model, dataset, tokenizer, device, name="XLM-R"):
    """Confusion matrix & per-class report, **excluding the “O” tag**."""
    id2label = model.config.id2label
    o_id     = {v: k for k, v in id2label.items()}["O"]

    all_true, all_pred = [], []

    for item, text in zip(dataset, dataset.texts):
        inputs = tokenizer(text,
                           truncation=True,
                           padding=True,
                           return_tensors="pt",
                           return_offsets_mapping=True)
        offsets = inputs.pop("offset_mapping")[0]
        inputs  = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            preds = torch.argmax(model(**inputs).logits, dim=-1)[0]

        for pred_id, true_id, (s, e) in zip(preds, item["labels"], offsets):
            if s == e or true_id == -100:      # skip specials / padding
                continue
            true_id, pred_id = int(true_id), int(pred_id)

            # drop tokens where **both** are “O”
            if true_id == o_id and pred_id == o_id:
                continue

            all_true.append(true_id)
            all_pred.append(pred_id)

    # unique labels w/out “O”
    uniq_ids   = sorted({i for i in all_true + all_pred if i != o_id})
    names      = [id2label[i] for i in uniq_ids]

    cm = confusion_matrix(all_true, all_pred, labels=uniq_ids)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
                xticklabels=names, yticklabels=names,
                cbar_kws=dict(label="Count"))
    plt.title(f"Confusion Matrix – {name} (no 'O')", fontsize=14)
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout(); plt.show()

    print(f"\n{name} Per-Class Metrics (no 'O')\n" + "="*60)
    print(classification_report(all_true, all_pred,
                                labels=uniq_ids,
                                target_names=names,
                                zero_division=0))
    return cm, all_true, all_pred


# 🔄 1.  Load checkpoint + tokenizer
xlmr_path      = "./hiligaynon-ner-xlm-roberta-final"   # or -deploy
tokenizer_xlmr = AutoTokenizer.from_pretrained(xlmr_path)
model_xlmr     = AutoModelForTokenClassification.from_pretrained(xlmr_path).to(device)
model_xlmr.eval()

# 🗂️ 2.  Build XLM-R-tokenised test set
test_dataset_xlmr = HiligaynonNERDataset(test_texts, test_entities, tokenizer_xlmr)

# 📊 3.  Generate confusion matrix (excludes “O”)
cm_xlmr, t_xlmr, p_xlmr = confusion_matrix_xlmr_no_O(
    model_xlmr, test_dataset_xlmr, tokenizer_xlmr, device, "XLM-R")


## Confusion Matrix Comparison

In [ ]:
# ============================================
# 📊 Comparative Analysis
# ============================================

def compare_models_performance(cm1, cm2, labels1, labels2, dataset):
    """
    Compare performance between mBERT and XLM-RoBERTa
    """
    # Get unique labels present in data
    unique_labels = sorted(list(set(labels1 + labels2)))
    label_names = [dataset.id_to_label[i] for i in unique_labels]

    # Calculate per-class accuracy
    print("\nPer-Entity Type Accuracy Comparison:")
    print("="*60)
    print(f"{'Entity Type':15} {'mBERT':>10} {'XLM-R':>10} {'Difference':>12}")
    print("-"*60)

    # Group by entity type
    entity_performance = {}

    for i, label_name in enumerate(label_names):
        if label_name == 'O':
            continue

        # Calculate accuracy for this label
        mbert_correct = np.diag(cm1)[i] if i < len(cm1) else 0
        mbert_total = np.sum(cm1[i, :]) if i < len(cm1) else 0
        mbert_acc = mbert_correct / mbert_total if mbert_total > 0 else 0

        xlmr_correct = np.diag(cm2)[i] if i < len(cm2) else 0
        xlmr_total = np.sum(cm2[i, :]) if i < len(cm2) else 0
        xlmr_acc = xlmr_correct / xlmr_total if xlmr_total > 0 else 0

        diff = xlmr_acc - mbert_acc

        print(f"{label_name:15} {mbert_acc:10.3f} {xlmr_acc:10.3f} {diff:+12.3f}")

        # Group by entity type
        entity_type = label_name.split('-')[1] if '-' in label_name else label_name
        if entity_type not in entity_performance:
            entity_performance[entity_type] = {'mbert': [], 'xlmr': []}
        entity_performance[entity_type]['mbert'].append(mbert_acc)
        entity_performance[entity_type]['xlmr'].append(xlmr_acc)

    # Average performance by entity type
    print("\nAverage Performance by Entity Type:")
    print("-"*60)
    print(f"{'Entity Type':15} {'mBERT':>10} {'XLM-R':>10} {'Winner':>12}")
    print("-"*60)

    for entity_type, scores in entity_performance.items():
        mbert_avg = np.mean(scores['mbert']) if scores['mbert'] else 0
        xlmr_avg = np.mean(scores['xlmr']) if scores['xlmr'] else 0
        winner = "mBERT" if mbert_avg > xlmr_avg else "XLM-R" if xlmr_avg > mbert_avg else "Tie"
        print(f"{entity_type:15} {mbert_avg:10.3f} {xlmr_avg:10.3f} {winner:>12}")

# Run comparison if both models are loaded
if 'cm_mbert' in globals() and 'cm_xlmr' in globals():
    compare_models_performance(cm_mbert, cm_xlmr, true_labels_mbert, true_labels_xlmr, test_dataset)

## Summary

This notebook demonstrates:
1. **Data Loading**: Loading Hiligaynon NER data from JSONL format
2. **Data Analysis**: Visualizing entity distribution and examples
3. **Preprocessing**: Converting character-level annotations to token-level labels
4. **Model Training**: Fine-tuning mBERT or XLM-RoBERTa for Hiligaynon NER
5. **Evaluation**: Computing precision, recall, F1, and accuracy metrics
6. **Inference**: Making predictions on new Hiligaynon text
7. **Error Analysis**: Understanding model mistakes
8. **Model Export**: Saving the model for deployment

### Next Steps:
1. **Data Augmentation**: Generate more training data using techniques like entity substitution
2. **Hyperparameter Tuning**: Optimize learning rate, batch size, and epochs
3. **Ensemble Methods**: Combine predictions from mBERT and XLM-RoBERTa
4. **Active Learning**: Identify and annotate difficult examples
5. **Cross-lingual Transfer**: Leverage related languages (Tagalog, Cebuano) for better performance

Zero-Shot Cross-lingual performance Hiligaynon to Cebuano
## XLM-RoBERTa


In [ ]:
import json
import torch
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification
)
from torch.utils.data import Dataset
import evaluate
from sklearn.metrics import classification_report, confusion_matrix
from tabulate import tabulate

# ===================================
# 1. Data Loading Functions
# ===================================

def load_cebuano_data(file_path):
    """Load Cebuano NER data and convert to Hiligaynon format"""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line.strip())
            # Convert Cebuano format to Hiligaynon format
            entities = []
            if 'label' in item and item['label']:
                for label_item in item['label']:
                    start = label_item['start']
                    end = label_item['end']
                    # Cebuano data only has B- tags, we'll use them as is
                    entity_label = label_item['labels'][0]
                    entities.append([start, end, entity_label])

            converted_item = {
                'text': item['text'],
                'entities': entities,
                'id': item.get('id', None)
            }
            data.append(converted_item)
    return data

def load_hiligaynon_data(file_path):
    """Load Hiligaynon NER data"""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

# ===================================
# 2. Dataset Class (Same as training)
# ===================================

class HiligaynonNERDataset(Dataset):
    """Custom dataset for NER - works for both Hiligaynon and Cebuano"""

    def __init__(self, texts, entities, tokenizer, max_length=128):
        self.texts = texts
        self.entities = entities
        self.tokenizer = tokenizer
        self.max_length = max_length

        # Define label mapping (same as training)
        self.label_to_id = {
            'O': 0,
            'B-PER': 1,
            'I-PER': 2,
            'B-LOC': 3,
            'I-LOC': 4,
            'B-ORG': 5,
            'I-ORG': 6,
            'B-DATE': 7,
            'I-DATE': 8
        }
        self.id_to_label = {v: k for k, v in self.label_to_id.items()}

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        entities = self.entities[idx]

        # Tokenize text
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_offsets_mapping=True,
            return_tensors='pt'
        )

        # Create labels based on character offsets
        labels = self._create_labels(text, entities, encoding['offset_mapping'][0])

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': labels
        }

    def _create_labels(self, text, entities, offset_mapping):
        """Create token-level labels based on character-level annotations"""
        labels = torch.zeros(len(offset_mapping), dtype=torch.long)

        # Sort entities by start position
        sorted_entities = sorted(entities, key=lambda x: x[0])

        for start_char, end_char, entity_label in sorted_entities:
            # Find tokens that overlap with entity
            for idx, (token_start, token_end) in enumerate(offset_mapping):
                if token_start == token_end:  # Special token
                    continue

                # Check if token overlaps with entity
                if token_start >= start_char and token_end <= end_char:
                    # For multi-token entities, we need to determine if this is B- or I-
                    if token_start == start_char:
                        # First token of entity - use the label as is (should be B-)
                        if entity_label in self.label_to_id:
                            labels[idx] = self.label_to_id[entity_label]
                    else:
                        # Subsequent tokens - convert B- to I-
                        if entity_label.startswith('B-'):
                            inside_label = 'I-' + entity_label[2:]
                            if inside_label in self.label_to_id:
                                labels[idx] = self.label_to_id[inside_label]
                        else:
                            # Use the label as is
                            if entity_label in self.label_to_id:
                                labels[idx] = self.label_to_id[entity_label]

        return labels

# ===================================
# 3. Evaluation Metrics
# ===================================

def compute_metrics(eval_pred):
    """Compute NER metrics using seqeval"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    # Convert predictions and labels to list of labels
    true_labels = []
    pred_labels = []

    # We'll need to get label_map from global scope
    label_map = LABEL_MAP  # Will be set globally

    for i in range(labels.shape[0]):
        true_label_seq = []
        pred_label_seq = []

        for j in range(labels.shape[1]):
            if labels[i, j] != -100:  # Ignore padding
                true_label_seq.append(label_map[labels[i, j]])
                pred_label_seq.append(label_map[predictions[i, j]])

        true_labels.append(true_label_seq)
        pred_labels.append(pred_label_seq)

    seqeval_metric = evaluate.load("seqeval")
    results = seqeval_metric.compute(
        predictions=pred_labels,
        references=true_labels,
        mode='strict',
        scheme='IOB2'
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# ===================================
# 4. Analysis Functions
# ===================================

def analyze_dataset_entities(data, dataset_name):
    """Analyze entity distribution in dataset"""
    entity_counts = Counter()
    entity_examples = {}

    for item in data:
        for start, end, label in item['entities']:
            entity_text = item['text'][start:end]
            # Normalize to entity type (remove B-/I- prefix)
            entity_type = label[2:] if label.startswith(('B-', 'I-')) else label
            entity_counts[entity_type] += 1

            if entity_type not in entity_examples:
                entity_examples[entity_type] = []
            if len(entity_examples[entity_type]) < 5:
                entity_examples[entity_type].append(entity_text)

    print(f"\n{'='*50}")
    print(f"Entity Distribution in {dataset_name}")
    print(f"{'='*50}")
    print(f"Total samples: {len(data)}")
    print(f"Total entities: {sum(entity_counts.values())}")
    print("\nEntity type counts:")
    for entity_type, count in entity_counts.most_common():
        print(f"  {entity_type}: {count}")
        print(f"    Examples: {', '.join(entity_examples[entity_type][:3])}")

    return entity_counts

def plot_entity_comparison(hiligaynon_counts, cebuano_counts):
    """Plot entity distribution comparison"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Hiligaynon distribution
    ax1.bar(hiligaynon_counts.keys(), hiligaynon_counts.values())
    ax1.set_title('Hiligaynon Entity Distribution')
    ax1.set_xlabel('Entity Type')
    ax1.set_ylabel('Count')

    # Cebuano distribution
    ax2.bar(cebuano_counts.keys(), cebuano_counts.values())
    ax2.set_title('Cebuano Entity Distribution')
    ax2.set_xlabel('Entity Type')
    ax2.set_ylabel('Count')

    plt.tight_layout()
    plt.show()

# ===================================
# 5. Main Evaluation Pipeline
# ===================================

def evaluate_crosslingual_transfer(
    model_path,
    hiligaynon_data_path,
    cebuano_data_path,
    model_type="xlm-roberta"
):
    """
    Main function to evaluate cross-lingual transfer
    """
    global LABEL_MAP

    print(f"\n{'='*60}")
    print(f"Zero-shot Cross-lingual Transfer Evaluation")
    print(f"Model: {model_type}")
    print(f"Source Language: Hiligaynon")
    print(f"Target Language: Cebuano")
    print(f"{'='*60}")

    # Load data
    print("\n1. Loading datasets...")
    hiligaynon_data = load_hiligaynon_data(hiligaynon_data_path)
    cebuano_data = load_cebuano_data(cebuano_data_path)

    # Analyze datasets
    print("\n2. Analyzing datasets...")
    hil_counts = analyze_dataset_entities(hiligaynon_data, "Hiligaynon")
    ceb_counts = analyze_dataset_entities(cebuano_data, "Cebuano")
    plot_entity_comparison(hil_counts, ceb_counts)

    # Load model and tokenizer
    print(f"\n3. Loading model from {model_path}...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Load tokenizer from base model to avoid tokenizer file issues
    tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
    model = AutoModelForTokenClassification.from_pretrained(model_path)
    model.to(device)

    # Prepare datasets
    print("\n4. Preparing datasets...")

    # Cebuano test set
    ceb_texts = [item['text'] for item in cebuano_data]
    ceb_entities = [item['entities'] for item in cebuano_data]
    cebuano_dataset = HiligaynonNERDataset(ceb_texts, ceb_entities, tokenizer)

    # For comparison, also prepare a Hiligaynon test set (last 20% of data)
    split_idx = int(0.8 * len(hiligaynon_data))
    hil_test_data = hiligaynon_data[split_idx:]
    hil_texts = [item['text'] for item in hil_test_data]
    hil_entities = [item['entities'] for item in hil_test_data]
    hiligaynon_dataset = HiligaynonNERDataset(hil_texts, hil_entities, tokenizer)

    # Set global label map for metrics
    LABEL_MAP = cebuano_dataset.id_to_label

    # Create data collator
    data_collator = DataCollatorForTokenClassification(
        tokenizer=tokenizer,
        pad_to_multiple_of=8,
        padding=True,
        label_pad_token_id=-100
    )

    # Training arguments (for evaluation only)
    training_args = TrainingArguments(
        output_dir="./eval_results",
        per_device_eval_batch_size=16,
        report_to="none",
        remove_unused_columns=False,
    )

    # Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    # Evaluate on both datasets
    print("\n5. Running evaluation...")

    print("\n   Evaluating on Hiligaynon test set...")
    hil_results = trainer.evaluate(eval_dataset=hiligaynon_dataset)

    print("\n   Evaluating on Cebuano (zero-shot)...")
    ceb_results = trainer.evaluate(eval_dataset=cebuano_dataset)

    # Display results comparison
    print("\n" + "="*60)
    print("EVALUATION RESULTS")
    print("="*60)

    results_df = pd.DataFrame({
        'Hiligaynon (in-language)': {
            'Precision': hil_results['eval_precision'],
            'Recall': hil_results['eval_recall'],
            'F1-Score': hil_results['eval_f1'],
            'Accuracy': hil_results['eval_accuracy']
        },
        'Cebuano (zero-shot)': {
            'Precision': ceb_results['eval_precision'],
            'Recall': ceb_results['eval_recall'],
            'F1-Score': ceb_results['eval_f1'],
            'Accuracy': ceb_results['eval_accuracy']
        }
    })

    print(tabulate(results_df.T, headers='keys', tablefmt='github', floatfmt='.4f'))

    # Calculate performance drop
    print("\n" + "="*60)
    print("CROSS-LINGUAL TRANSFER ANALYSIS")
    print("="*60)

    f1_drop = ((hil_results['eval_f1'] - ceb_results['eval_f1']) / hil_results['eval_f1']) * 100
    precision_drop = ((hil_results['eval_precision'] - ceb_results['eval_precision']) / hil_results['eval_precision']) * 100
    recall_drop = ((hil_results['eval_recall'] - ceb_results['eval_recall']) / hil_results['eval_recall']) * 100

    print(f"F1-Score drop: {f1_drop:.2f}%")
    print(f"Precision drop: {precision_drop:.2f}%")
    print(f"Recall drop: {recall_drop:.2f}%")

    # Visualize results
    fig, ax = plt.subplots(figsize=(10, 6))
    metrics = ['Precision', 'Recall', 'F1-Score', 'Accuracy']
    x = np.arange(len(metrics))
    width = 0.35

    hil_scores = [hil_results['eval_precision'], hil_results['eval_recall'],
                  hil_results['eval_f1'], hil_results['eval_accuracy']]
    ceb_scores = [ceb_results['eval_precision'], ceb_results['eval_recall'],
                  ceb_results['eval_f1'], ceb_results['eval_accuracy']]

    rects1 = ax.bar(x - width/2, hil_scores, width, label='Hiligaynon')
    rects2 = ax.bar(x + width/2, ceb_scores, width, label='Cebuano')

    ax.set_ylabel('Score')
    ax.set_title('Cross-lingual Transfer Performance: Hiligaynon → Cebuano')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend()
    ax.set_ylim(0, 1.1)

    # Add value labels on bars
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.3f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha='center', va='bottom')

    autolabel(rects1)
    autolabel(rects2)

    plt.tight_layout()
    plt.show()

    return results_df, hil_results, ceb_results

# ===================================
# 6. Error Analysis
# ===================================

def perform_error_analysis(model, tokenizer, test_data, num_samples=10):
    """Analyze prediction errors on sample sentences"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()

    print(f"\n{'='*60}")
    print(f"ERROR ANALYSIS (First {num_samples} samples)")
    print(f"{'='*60}")

    for i, item in enumerate(test_data[:num_samples]):
        text = item['text']
        true_entities = item['entities']

        # Get predictions
        inputs = tokenizer(
            text,
            truncation=True,
            padding=True,
            return_tensors="pt",
            return_offsets_mapping=True
        )

        offset_mapping = inputs.pop('offset_mapping')[0]
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.argmax(outputs.logits, dim=-1)[0]

        # Convert predictions to entities
        predicted_entities = []
        label_map = {0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-LOC', 4: 'I-LOC',
                     5: 'B-ORG', 6: 'I-ORG', 7: 'B-DATE', 8: 'I-DATE'}

        current_entity = None
        for idx, (pred_id, (start, end)) in enumerate(zip(predictions, offset_mapping)):
            if start == end:  # Special token
                continue

            label = label_map[pred_id.item()]

            if label.startswith('B-'):
                if current_entity:
                    predicted_entities.append(current_entity)
                current_entity = {
                    'text': text[start:end],
                    'label': label,
                    'start': start.item(),
                    'end': end.item()
                }
            elif label.startswith('I-') and current_entity:
                current_entity['text'] = text[current_entity['start']:end]
                current_entity['end'] = end.item()
            else:
                if current_entity:
                    predicted_entities.append(current_entity)
                    current_entity = None

        if current_entity:
            predicted_entities.append(current_entity)

        # Display results
        print(f"\nSample {i+1}:")
        print(f"Text: {text}")
        print(f"True entities: {true_entities}")
        print(f"Predicted: {[(e['text'], e['label']) for e in predicted_entities]}")

# ===================================
# 7. Run Evaluation
# ===================================

if __name__ == "__main__":
    # Configuration - Updated for XLM-RoBERTa
    MODEL_PATH = "hiligaynon-ner-xlm-roberta-deploy"
    HILIGAYNON_DATA = "final_combined_all.jsonl"
    CEBUANO_DATA = "cebuano_converted.jsonl"

    # Run evaluation
    results_df, hil_results, ceb_results = evaluate_crosslingual_transfer(
        model_path=MODEL_PATH,
        hiligaynon_data_path=HILIGAYNON_DATA,
        cebuano_data_path=CEBUANO_DATA,
        model_type="xlm-roberta"
    )

    # Optional: Perform error analysis on Cebuano data
    print("\n" + "="*60)
    print("Would you like to perform error analysis? (Showing sample predictions)")

    # Load model and data for error analysis
    cebuano_data = load_cebuano_data(CEBUANO_DATA)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    perform_error_analysis(model, tokenizer, cebuano_data, num_samples=5)

## mBERT Hiligaynon to Cebuano

In [ ]:
import json
import torch
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification
)
from torch.utils.data import Dataset
import evaluate
from sklearn.metrics import classification_report, confusion_matrix
from tabulate import tabulate

# ===================================
# 1. Data Loading Functions
# ===================================

def load_cebuano_data(file_path):
    """Load Cebuano NER data and convert to Hiligaynon format"""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line.strip())
            # Convert Cebuano format to Hiligaynon format
            entities = []
            if 'label' in item and item['label']:
                for label_item in item['label']:
                    start = label_item['start']
                    end = label_item['end']
                    # Cebuano data only has B- tags, we'll use them as is
                    entity_label = label_item['labels'][0]
                    entities.append([start, end, entity_label])

            converted_item = {
                'text': item['text'],
                'entities': entities,
                'id': item.get('id', None)
            }
            data.append(converted_item)
    return data

def load_hiligaynon_data(file_path):
    """Load Hiligaynon NER data"""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

# ===================================
# 2. Dataset Class (Same as training)
# ===================================

class HiligaynonNERDataset(Dataset):
    """Custom dataset for NER - works for both Hiligaynon and Cebuano"""

    def __init__(self, texts, entities, tokenizer, max_length=128):
        self.texts = texts
        self.entities = entities
        self.tokenizer = tokenizer
        self.max_length = max_length

        # Define label mapping (same as training)
        self.label_to_id = {
            'O': 0,
            'B-PER': 1,
            'I-PER': 2,
            'B-LOC': 3,
            'I-LOC': 4,
            'B-ORG': 5,
            'I-ORG': 6,
            'B-DATE': 7,
            'I-DATE': 8
        }
        self.id_to_label = {v: k for k, v in self.label_to_id.items()}

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        entities = self.entities[idx]

        # Tokenize text
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_offsets_mapping=True,
            return_tensors='pt'
        )

        # Create labels based on character offsets
        labels = self._create_labels(text, entities, encoding['offset_mapping'][0])

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': labels
        }

    def _create_labels(self, text, entities, offset_mapping):
        """Create token-level labels based on character-level annotations"""
        labels = torch.zeros(len(offset_mapping), dtype=torch.long)

        # Sort entities by start position
        sorted_entities = sorted(entities, key=lambda x: x[0])

        for start_char, end_char, entity_label in sorted_entities:
            # Find tokens that overlap with entity
            for idx, (token_start, token_end) in enumerate(offset_mapping):
                if token_start == token_end:  # Special token
                    continue

                # Check if token overlaps with entity
                if token_start >= start_char and token_end <= end_char:
                    # For multi-token entities, we need to determine if this is B- or I-
                    if token_start == start_char:
                        # First token of entity - use the label as is (should be B-)
                        if entity_label in self.label_to_id:
                            labels[idx] = self.label_to_id[entity_label]
                    else:
                        # Subsequent tokens - convert B- to I-
                        if entity_label.startswith('B-'):
                            inside_label = 'I-' + entity_label[2:]
                            if inside_label in self.label_to_id:
                                labels[idx] = self.label_to_id[inside_label]
                        else:
                            # Use the label as is
                            if entity_label in self.label_to_id:
                                labels[idx] = self.label_to_id[entity_label]

        return labels

# ===================================
# 3. Evaluation Metrics
# ===================================

def compute_metrics(eval_pred):
    """Compute NER metrics using seqeval"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    # Convert predictions and labels to list of labels
    true_labels = []
    pred_labels = []

    # We'll need to get label_map from global scope
    label_map = LABEL_MAP  # Will be set globally

    for i in range(labels.shape[0]):
        true_label_seq = []
        pred_label_seq = []

        for j in range(labels.shape[1]):
            if labels[i, j] != -100:  # Ignore padding
                true_label_seq.append(label_map[labels[i, j]])
                pred_label_seq.append(label_map[predictions[i, j]])

        true_labels.append(true_label_seq)
        pred_labels.append(pred_label_seq)

    seqeval_metric = evaluate.load("seqeval")
    results = seqeval_metric.compute(
        predictions=pred_labels,
        references=true_labels,
        mode='strict',
        scheme='IOB2'
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# ===================================
# 4. Analysis Functions
# ===================================

def analyze_dataset_entities(data, dataset_name):
    """Analyze entity distribution in dataset"""
    entity_counts = Counter()
    entity_examples = {}

    for item in data:
        for start, end, label in item['entities']:
            entity_text = item['text'][start:end]
            # Normalize to entity type (remove B-/I- prefix)
            entity_type = label[2:] if label.startswith(('B-', 'I-')) else label
            entity_counts[entity_type] += 1

            if entity_type not in entity_examples:
                entity_examples[entity_type] = []
            if len(entity_examples[entity_type]) < 5:
                entity_examples[entity_type].append(entity_text)

    print(f"\n{'='*50}")
    print(f"Entity Distribution in {dataset_name}")
    print(f"{'='*50}")
    print(f"Total samples: {len(data)}")
    print(f"Total entities: {sum(entity_counts.values())}")
    print("\nEntity type counts:")
    for entity_type, count in entity_counts.most_common():
        print(f"  {entity_type}: {count}")
        print(f"    Examples: {', '.join(entity_examples[entity_type][:3])}")

    return entity_counts

def plot_entity_comparison(hiligaynon_counts, cebuano_counts):
    """Plot entity distribution comparison"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Hiligaynon distribution
    ax1.bar(hiligaynon_counts.keys(), hiligaynon_counts.values())
    ax1.set_title('Hiligaynon Entity Distribution')
    ax1.set_xlabel('Entity Type')
    ax1.set_ylabel('Count')

    # Cebuano distribution
    ax2.bar(cebuano_counts.keys(), cebuano_counts.values())
    ax2.set_title('Cebuano Entity Distribution')
    ax2.set_xlabel('Entity Type')
    ax2.set_ylabel('Count')

    plt.tight_layout()
    plt.show()

# ===================================
# 5. Main Evaluation Pipeline
# ===================================

def evaluate_crosslingual_transfer(
    model_path,
    hiligaynon_data_path,
    cebuano_data_path,
    model_type="mbert"
):
    """
    Main function to evaluate cross-lingual transfer
    """
    global LABEL_MAP

    print(f"\n{'='*60}")
    print(f"Zero-shot Cross-lingual Transfer Evaluation")
    print(f"Model: {model_type}")
    print(f"Source Language: Hiligaynon")
    print(f"Target Language: Cebuano")
    print(f"{'='*60}")

    # Load data
    print("\n1. Loading datasets...")
    hiligaynon_data = load_hiligaynon_data(hiligaynon_data_path)
    cebuano_data = load_cebuano_data(cebuano_data_path)

    # Analyze datasets
    print("\n2. Analyzing datasets...")
    hil_counts = analyze_dataset_entities(hiligaynon_data, "Hiligaynon")
    ceb_counts = analyze_dataset_entities(cebuano_data, "Cebuano")
    plot_entity_comparison(hil_counts, ceb_counts)

    # Load model and tokenizer
    print(f"\n3. Loading model from {model_path}...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForTokenClassification.from_pretrained(model_path)
    model.to(device)

    # Prepare datasets
    print("\n4. Preparing datasets...")

    # Cebuano test set
    ceb_texts = [item['text'] for item in cebuano_data]
    ceb_entities = [item['entities'] for item in cebuano_data]
    cebuano_dataset = HiligaynonNERDataset(ceb_texts, ceb_entities, tokenizer)

    # For comparison, also prepare a Hiligaynon test set (last 20% of data)
    split_idx = int(0.8 * len(hiligaynon_data))
    hil_test_data = hiligaynon_data[split_idx:]
    hil_texts = [item['text'] for item in hil_test_data]
    hil_entities = [item['entities'] for item in hil_test_data]
    hiligaynon_dataset = HiligaynonNERDataset(hil_texts, hil_entities, tokenizer)

    # Set global label map for metrics
    LABEL_MAP = cebuano_dataset.id_to_label

    # Create data collator
    data_collator = DataCollatorForTokenClassification(
        tokenizer=tokenizer,
        pad_to_multiple_of=8,
        padding=True,
        label_pad_token_id=-100
    )

    # Training arguments (for evaluation only)
    training_args = TrainingArguments(
        output_dir="./eval_results",
        per_device_eval_batch_size=16,
        report_to="none",
        remove_unused_columns=False,
    )

    # Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    # Evaluate on both datasets
    print("\n5. Running evaluation...")

    print("\n   Evaluating on Hiligaynon test set...")
    hil_results = trainer.evaluate(eval_dataset=hiligaynon_dataset)

    print("\n   Evaluating on Cebuano (zero-shot)...")
    ceb_results = trainer.evaluate(eval_dataset=cebuano_dataset)

    # Display results comparison
    print("\n" + "="*60)
    print("EVALUATION RESULTS")
    print("="*60)

    results_df = pd.DataFrame({
        'Hiligaynon (in-language)': {
            'Precision': hil_results['eval_precision'],
            'Recall': hil_results['eval_recall'],
            'F1-Score': hil_results['eval_f1'],
            'Accuracy': hil_results['eval_accuracy']
        },
        'Cebuano (zero-shot)': {
            'Precision': ceb_results['eval_precision'],
            'Recall': ceb_results['eval_recall'],
            'F1-Score': ceb_results['eval_f1'],
            'Accuracy': ceb_results['eval_accuracy']
        }
    })

    print(tabulate(results_df.T, headers='keys', tablefmt='github', floatfmt='.4f'))

    # Calculate performance drop
    print("\n" + "="*60)
    print("CROSS-LINGUAL TRANSFER ANALYSIS")
    print("="*60)

    f1_drop = ((hil_results['eval_f1'] - ceb_results['eval_f1']) / hil_results['eval_f1']) * 100
    precision_drop = ((hil_results['eval_precision'] - ceb_results['eval_precision']) / hil_results['eval_precision']) * 100
    recall_drop = ((hil_results['eval_recall'] - ceb_results['eval_recall']) / hil_results['eval_recall']) * 100

    print(f"F1-Score drop: {f1_drop:.2f}%")
    print(f"Precision drop: {precision_drop:.2f}%")
    print(f"Recall drop: {recall_drop:.2f}%")

    # Visualize results
    fig, ax = plt.subplots(figsize=(10, 6))
    metrics = ['Precision', 'Recall', 'F1-Score', 'Accuracy']
    x = np.arange(len(metrics))
    width = 0.35

    hil_scores = [hil_results['eval_precision'], hil_results['eval_recall'],
                  hil_results['eval_f1'], hil_results['eval_accuracy']]
    ceb_scores = [ceb_results['eval_precision'], ceb_results['eval_recall'],
                  ceb_results['eval_f1'], ceb_results['eval_accuracy']]

    rects1 = ax.bar(x - width/2, hil_scores, width, label='Hiligaynon')
    rects2 = ax.bar(x + width/2, ceb_scores, width, label='Cebuano')

    ax.set_ylabel('Score')
    ax.set_title('Cross-lingual Transfer Performance: Hiligaynon → Cebuano')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend()
    ax.set_ylim(0, 1.1)

    # Add value labels on bars
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.3f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha='center', va='bottom')

    autolabel(rects1)
    autolabel(rects2)

    plt.tight_layout()
    plt.show()

    return results_df, hil_results, ceb_results

# ===================================
# 6. Error Analysis
# ===================================

def perform_error_analysis(model, tokenizer, test_data, num_samples=10):
    """Analyze prediction errors on sample sentences"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()

    print(f"\n{'='*60}")
    print(f"ERROR ANALYSIS (First {num_samples} samples)")
    print(f"{'='*60}")

    for i, item in enumerate(test_data[:num_samples]):
        text = item['text']
        true_entities = item['entities']

        # Get predictions
        inputs = tokenizer(
            text,
            truncation=True,
            padding=True,
            return_tensors="pt",
            return_offsets_mapping=True
        )

        offset_mapping = inputs.pop('offset_mapping')[0]
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.argmax(outputs.logits, dim=-1)[0]

        # Convert predictions to entities
        predicted_entities = []
        label_map = {0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-LOC', 4: 'I-LOC',
                     5: 'B-ORG', 6: 'I-ORG', 7: 'B-DATE', 8: 'I-DATE'}

        current_entity = None
        for idx, (pred_id, (start, end)) in enumerate(zip(predictions, offset_mapping)):
            if start == end:  # Special token
                continue

            label = label_map[pred_id.item()]

            if label.startswith('B-'):
                if current_entity:
                    predicted_entities.append(current_entity)
                current_entity = {
                    'text': text[start:end],
                    'label': label,
                    'start': start.item(),
                    'end': end.item()
                }
            elif label.startswith('I-') and current_entity:
                current_entity['text'] = text[current_entity['start']:end]
                current_entity['end'] = end.item()
            else:
                if current_entity:
                    predicted_entities.append(current_entity)
                    current_entity = None

        if current_entity:
            predicted_entities.append(current_entity)

        # Display results
        print(f"\nSample {i+1}:")
        print(f"Text: {text}")
        print(f"True entities: {true_entities}")
        print(f"Predicted: {[(e['text'], e['label']) for e in predicted_entities]}")

# ===================================
# 7. Run Evaluation
# ===================================

if __name__ == "__main__":
    # Configuration
    MODEL_PATH = "hiligaynon-ner-mbert-deploy"
    HILIGAYNON_DATA = "final_combined_all.jsonl"
    CEBUANO_DATA = "cebuano_converted.jsonl"

    # Run evaluation
    results_df, hil_results, ceb_results = evaluate_crosslingual_transfer(
        model_path=MODEL_PATH,
        hiligaynon_data_path=HILIGAYNON_DATA,
        cebuano_data_path=CEBUANO_DATA,
        model_type="mbert"
    )

    # Optional: Perform error analysis on Cebuano data
    print("\n" + "="*60)
    print("Would you like to perform error analysis? (Showing sample predictions)")

    # Load model and data for error analysis
    cebuano_data = load_cebuano_data(CEBUANO_DATA)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    perform_error_analysis(model, tokenizer, cebuano_data, num_samples=5)

## Zero-Shot Cross-lingual Performance : Hiligaynon to Tagalog

In [ ]:
import json
import torch
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification
)
from torch.utils.data import Dataset
import evaluate
from sklearn.metrics import classification_report, confusion_matrix
from tabulate import tabulate

# ===================================
# 1. Data Loading Functions
# ===================================

def load_tagalog_data(file_path):
    """
    Load the converted Tagalog data.

    It first looks for the modern 'entities' key (list of [start, end, tag]).
    If that is absent but the old 'label' key is present, it falls back to the
    conversion logic you already wrote.
    """
    data = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line.strip())

            # --- NEW LOGIC --------------------------------------------------
            if 'entities' in item and item['entities']:
                entities = item['entities']          # already converted ✔
            # ----------------------------------------------------------------
            elif 'label' in item and item['label']:  # legacy format fallback
                entities = []
                for lbl in item['label']:
                    entities.append([lbl['start'], lbl['end'], lbl['labels'][0]])
            else:
                entities = []

            data.append({
                'text': item['text'],
                'entities': entities,
                'id': item.get('id')
            })

    return data


def load_hiligaynon_data(file_path):
    """Load Hiligaynon NER data"""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

# ===================================
# 2. Dataset Class (Same as training)
# ===================================

class HiligaynonNERDataset(Dataset):
    """Custom dataset for NER - works for both Hiligaynon and Tagalog"""

    def __init__(self, texts, entities, tokenizer, max_length=128):
        self.texts = texts
        self.entities = entities
        self.tokenizer = tokenizer
        self.max_length = max_length

        # Define label mapping (same as training)
        self.label_to_id = {
            'O': 0,
            'B-PER': 1,
            'I-PER': 2,
            'B-LOC': 3,
            'I-LOC': 4,
            'B-ORG': 5,
            'I-ORG': 6,
            'B-DATE': 7,
            'I-DATE': 8
        }
        self.id_to_label = {v: k for k, v in self.label_to_id.items()}

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        entities = self.entities[idx]

        # Tokenize text
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_offsets_mapping=True,
            return_tensors='pt'
        )

        # Create labels based on character offsets
        labels = self._create_labels(text, entities, encoding['offset_mapping'][0])

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': labels
        }

    def _create_labels(self, text, entities, offset_mapping):
        """Create token-level labels based on character-level annotations"""
        labels = torch.zeros(len(offset_mapping), dtype=torch.long)

        # Sort entities by start position
        sorted_entities = sorted(entities, key=lambda x: x[0])

        for start_char, end_char, entity_label in sorted_entities:
            # Find tokens that overlap with entity
            for idx, (token_start, token_end) in enumerate(offset_mapping):
                if token_start == token_end:  # Special token
                    continue

                # Check if token overlaps with entity
                if token_start >= start_char and token_end <= end_char:
                    # For multi-token entities, we need to determine if this is B- or I-
                    if token_start == start_char:
                        # First token of entity - use the label as is (should be B-)
                        if entity_label in self.label_to_id:
                            labels[idx] = self.label_to_id[entity_label]
                    else:
                        # Subsequent tokens - convert B- to I-
                        if entity_label.startswith('B-'):
                            inside_label = 'I-' + entity_label[2:]
                            if inside_label in self.label_to_id:
                                labels[idx] = self.label_to_id[inside_label]
                        else:
                            # Use the label as is
                            if entity_label in self.label_to_id:
                                labels[idx] = self.label_to_id[entity_label]

        return labels

# ===================================
# 3. Evaluation Metrics
# ===================================

def compute_metrics(eval_pred):
    """Compute NER metrics using seqeval"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    # Convert predictions and labels to list of labels
    true_labels = []
    pred_labels = []

    # We'll need to get label_map from global scope
    label_map = LABEL_MAP  # Will be set globally

    for i in range(labels.shape[0]):
        true_label_seq = []
        pred_label_seq = []

        for j in range(labels.shape[1]):
            if labels[i, j] != -100:  # Ignore padding
                true_label_seq.append(label_map[labels[i, j]])
                pred_label_seq.append(label_map[predictions[i, j]])

        true_labels.append(true_label_seq)
        pred_labels.append(pred_label_seq)

    seqeval_metric = evaluate.load("seqeval")
    results = seqeval_metric.compute(
        predictions=pred_labels,
        references=true_labels,
        mode='strict',
        scheme='IOB2'
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# ===================================
# 4. Analysis Functions
# ===================================

def analyze_dataset_entities(data, dataset_name):
    """Analyze entity distribution in dataset"""
    entity_counts = Counter()
    entity_examples = {}

    for item in data:
        for start, end, label in item['entities']:
            entity_text = item['text'][start:end]
            # Normalize to entity type (remove B-/I- prefix)
            entity_type = label[2:] if label.startswith(('B-', 'I-')) else label
            entity_counts[entity_type] += 1

            if entity_type not in entity_examples:
                entity_examples[entity_type] = []
            if len(entity_examples[entity_type]) < 5:
                entity_examples[entity_type].append(entity_text)

    print(f"\n{'='*50}")
    print(f"Entity Distribution in {dataset_name}")
    print(f"{'='*50}")
    print(f"Total samples: {len(data)}")
    print(f"Total entities: {sum(entity_counts.values())}")
    print("\nEntity type counts:")
    for entity_type, count in entity_counts.most_common():
        print(f"  {entity_type}: {count}")
        print(f"    Examples: {', '.join(entity_examples[entity_type][:3])}")

    return entity_counts

def plot_entity_comparison(hiligaynon_counts, tagalog_counts):
    """Plot entity distribution comparison"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Hiligaynon distribution
    ax1.bar(hiligaynon_counts.keys(), hiligaynon_counts.values())
    ax1.set_title('Hiligaynon Entity Distribution')
    ax1.set_xlabel('Entity Type')
    ax1.set_ylabel('Count')

    # Tagalog distribution
    ax2.bar(tagalog_counts.keys(), tagalog_counts.values())
    ax2.set_title('Tagalog Entity Distribution')
    ax2.set_xlabel('Entity Type')
    ax2.set_ylabel('Count')

    plt.tight_layout()
    plt.show()

# ===================================
# 5. Main Evaluation Pipeline
# ===================================

def evaluate_crosslingual_transfer(
    model_path,
    hiligaynon_data_path,
    tagalog_data_path,
    model_type="mbert"
):
    """
    Main function to evaluate cross-lingual transfer
    """
    global LABEL_MAP

    print(f"\n{'='*60}")
    print(f"Zero-shot Cross-lingual Transfer Evaluation")
    print(f"Model: {model_type}")
    print(f"Source Language: Hiligaynon")
    print(f"Target Language: Tagalog")
    print(f"{'='*60}")

    # Load data
    print("\n1. Loading datasets...")
    hiligaynon_data = load_hiligaynon_data(hiligaynon_data_path)
    tagalog_data = load_tagalog_data(tagalog_data_path)

    # Analyze datasets
    print("\n2. Analyzing datasets...")
    hil_counts = analyze_dataset_entities(hiligaynon_data, "Hiligaynon")
    tag_counts = analyze_dataset_entities(tagalog_data, "Tagalog")
    plot_entity_comparison(hil_counts, tag_counts)

    # Load model and tokenizer
    print(f"\n3. Loading model from {model_path}...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForTokenClassification.from_pretrained(model_path)
    model.to(device)

    # Prepare datasets
    print("\n4. Preparing datasets...")

    # Tagalog test set
    tag_texts = [item['text'] for item in tagalog_data]
    tag_entities = [item['entities'] for item in tagalog_data]
    tagalog_dataset = HiligaynonNERDataset(tag_texts, tag_entities, tokenizer)

    # For comparison, also prepare a Hiligaynon test set (last 20% of data)
    split_idx = int(0.8 * len(hiligaynon_data))
    hil_test_data = hiligaynon_data[split_idx:]
    hil_texts = [item['text'] for item in hil_test_data]
    hil_entities = [item['entities'] for item in hil_test_data]
    hiligaynon_dataset = HiligaynonNERDataset(hil_texts, hil_entities, tokenizer)

    # Set global label map for metrics
    LABEL_MAP = tagalog_dataset.id_to_label

    # Create data collator
    data_collator = DataCollatorForTokenClassification(
        tokenizer=tokenizer,
        pad_to_multiple_of=8,
        padding=True,
        label_pad_token_id=-100
    )

    # Training arguments (for evaluation only)
    training_args = TrainingArguments(
        output_dir="./eval_results",
        per_device_eval_batch_size=16,
        report_to="none",
        remove_unused_columns=False,
    )

    # Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    # Evaluate on both datasets
    print("\n5. Running evaluation...")

    print("\n   Evaluating on Hiligaynon test set...")
    hil_results = trainer.evaluate(eval_dataset=hiligaynon_dataset)

    print("\n   Evaluating on Tagalog (zero-shot)...")
    tag_results = trainer.evaluate(eval_dataset=tagalog_dataset)

    # Display results comparison
    print("\n" + "="*60)
    print("EVALUATION RESULTS")
    print("="*60)

    results_df = pd.DataFrame({
        'Hiligaynon (in-language)': {
            'Precision': hil_results['eval_precision'],
            'Recall': hil_results['eval_recall'],
            'F1-Score': hil_results['eval_f1'],
            'Accuracy': hil_results['eval_accuracy']
        },
        'Tagalog (zero-shot)': {
            'Precision': tag_results['eval_precision'],
            'Recall': tag_results['eval_recall'],
            'F1-Score': tag_results['eval_f1'],
            'Accuracy': tag_results['eval_accuracy']
        }
    })

    print(tabulate(results_df.T, headers='keys', tablefmt='github', floatfmt='.4f'))

    # Calculate performance drop
    print("\n" + "="*60)
    print("CROSS-LINGUAL TRANSFER ANALYSIS")
    print("="*60)

    f1_drop = ((hil_results['eval_f1'] - tag_results['eval_f1']) / hil_results['eval_f1']) * 100
    precision_drop = ((hil_results['eval_precision'] - tag_results['eval_precision']) / hil_results['eval_precision']) * 100
    recall_drop = ((hil_results['eval_recall'] - tag_results['eval_recall']) / hil_results['eval_recall']) * 100

    print(f"F1-Score drop: {f1_drop:.2f}%")
    print(f"Precision drop: {precision_drop:.2f}%")
    print(f"Recall drop: {recall_drop:.2f}%")

    # Visualize results
    fig, ax = plt.subplots(figsize=(10, 6))
    metrics = ['Precision', 'Recall', 'F1-Score', 'Accuracy']
    x = np.arange(len(metrics))
    width = 0.35

    hil_scores = [hil_results['eval_precision'], hil_results['eval_recall'],
                  hil_results['eval_f1'], hil_results['eval_accuracy']]
    tag_scores = [tag_results['eval_precision'], tag_results['eval_recall'],
                  tag_results['eval_f1'], tag_results['eval_accuracy']]

    rects1 = ax.bar(x - width/2, hil_scores, width, label='Hiligaynon')
    rects2 = ax.bar(x + width/2, tag_scores, width, label='Tagalog')

    ax.set_ylabel('Score')
    ax.set_title('Cross-lingual Transfer Performance: Hiligaynon → Tagalog')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend()
    ax.set_ylim(0, 1.1)

    # Add value labels on bars
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.3f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha='center', va='bottom')

    autolabel(rects1)
    autolabel(rects2)

    plt.tight_layout()
    plt.show()

    return results_df, hil_results, tag_results

# ===================================
# 6. Error Analysis
# ===================================

def perform_error_analysis(model, tokenizer, test_data, num_samples=10):
    """Analyze prediction errors on sample sentences"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()

    print(f"\n{'='*60}")
    print(f"ERROR ANALYSIS (First {num_samples} samples)")
    print(f"{'='*60}")

    for i, item in enumerate(test_data[:num_samples]):
        text = item['text']
        true_entities = item['entities']

        # Get predictions
        inputs = tokenizer(
            text,
            truncation=True,
            padding=True,
            return_tensors="pt",
            return_offsets_mapping=True
        )

        offset_mapping = inputs.pop('offset_mapping')[0]
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.argmax(outputs.logits, dim=-1)[0]

        # Convert predictions to entities
        predicted_entities = []
        label_map = {0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-LOC', 4: 'I-LOC',
                     5: 'B-ORG', 6: 'I-ORG', 7: 'B-DATE', 8: 'I-DATE'}

        current_entity = None
        for idx, (pred_id, (start, end)) in enumerate(zip(predictions, offset_mapping)):
            if start == end:  # Special token
                continue

            label = label_map[pred_id.item()]

            if label.startswith('B-'):
                if current_entity:
                    predicted_entities.append(current_entity)
                current_entity = {
                    'text': text[start:end],
                    'label': label,
                    'start': start.item(),
                    'end': end.item()
                }
            elif label.startswith('I-') and current_entity:
                current_entity['text'] = text[current_entity['start']:end]
                current_entity['end'] = end.item()
            else:
                if current_entity:
                    predicted_entities.append(current_entity)
                    current_entity = None

        if current_entity:
            predicted_entities.append(current_entity)

        # Display results
        print(f"\nSample {i+1}:")
        print(f"Text: {text}")
        print(f"True entities: {true_entities}")
        print(f"Predicted: {[(e['text'], e['label']) for e in predicted_entities]}")

# ===================================
# 7. Run Evaluation
# ===================================

if __name__ == "__main__":
    # Configuration
    MODEL_PATH = "hiligaynon-ner-mbert-deploy"
    HILIGAYNON_DATA = "final_combined_all.jsonl"
    TAGALOG_DATA = "processed_tagalog_dataset_converted.jsonl"

    # Run evaluation
    results_df, hil_results, tag_results = evaluate_crosslingual_transfer(
        model_path=MODEL_PATH,
        hiligaynon_data_path=HILIGAYNON_DATA,
        tagalog_data_path=TAGALOG_DATA,
        model_type="mbert"
    )

    # Optional: Perform error analysis on Tagalog data
    print("\n" + "="*60)
    print("Would you like to perform error analysis? (Showing sample predictions)")

    # Load model and data for error analysis
    tagalog_data = load_tagalog_data(TAGALOG_DATA)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    perform_error_analysis(model, tokenizer, tagalog_data, num_samples=5)

In [ ]:
tagalog_data = load_tagalog_data("processed_tagalog_dataset_converted.jsonl")
analyze_dataset_entities(tagalog_data, "Tagalog")


## XLM-RoBERTa Hiligaynon to Tagalog


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
cross_lingual_hil_to_tgl.py
---------------------------

Evaluate zero-shot transfer from a Hiligaynon NER model to Tagalog.
Assumes the Tagalog data are already converted to JSONL with full
B-/I- prefixes (e.g. processed_tagalog_dataset_converted.jsonl).

Model: XLM-RoBERTa fine-tuned on Hiligaynon.
"""

# --------------------------------------------------------------
# 0. Imports and global settings
# --------------------------------------------------------------
import json
import warnings
from collections import Counter
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset
from tabulate import tabulate
import evaluate

from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification
)

warnings.filterwarnings("ignore")


# --------------------------------------------------------------
# 1. Data-loading helpers
# --------------------------------------------------------------
def load_hiligaynon_data(path: str) -> List[Dict]:
    """Each line: {"text": ..., "entities": [[start,end,label], ...]}"""
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def load_tagalog_data(path: str) -> List[Dict]:
    """Accepts either new 'entities' key or legacy 'label' key."""
    data = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            item = json.loads(line.strip())

            if "entities" in item and item["entities"]:
                entities = item["entities"]
            elif "label" in item and item["label"]:
                entities = [
                    [lbl["start"], lbl["end"], lbl["labels"][0]]
                    for lbl in item["label"]
                ]
            else:
                entities = []

            data.append(
                {
                    "text": item["text"],
                    "entities": entities,
                    "id": item.get("id"),
                }
            )
    return data


# --------------------------------------------------------------
# 2. Dataset class (same one for both languages)
# --------------------------------------------------------------
class HiligaynonNERDataset(Dataset):
    def __init__(
        self,
        texts: List[str],
        entities: List[List[Tuple[int, int, str]]],
        tokenizer,
        max_length: int = 128,
    ):
        self.texts = texts
        self.entities = entities
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.label_to_id = {
            "O": 0,
            "B-PER": 1,
            "I-PER": 2,
            "B-LOC": 3,
            "I-LOC": 4,
            "B-ORG": 5,
            "I-ORG": 6,
            "B-DATE": 7,
            "I-DATE": 8,
        }
        self.id_to_label = {v: k for k, v in self.label_to_id.items()}

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        ents = self.entities[idx]

        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_offsets_mapping=True,
            return_tensors="pt",
        )

        labels = self._char_to_token_labels(text, ents, enc["offset_mapping"][0])
        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels": labels,
        }

    # --- helper -------------------------------------------------
    def _char_to_token_labels(self, text, entities, offset_mapping):
        labels = torch.zeros(len(offset_mapping), dtype=torch.long)
        for start_c, end_c, tag in sorted(entities, key=lambda x: x[0]):
            for i, (tok_start, tok_end) in enumerate(offset_mapping):
                if tok_start == tok_end:  # special tokens
                    continue
                if tok_start < end_c and tok_end > start_c:  # any overlap
                    if tok_start == start_c and tag in self.label_to_id:
                        labels[i] = self.label_to_id[tag]  # B- tag
                    elif tag.startswith("B-"):
                        inside = "I-" + tag[2:]
                        labels[i] = self.label_to_id.get(inside, 0)
                    else:
                        labels[i] = self.label_to_id.get(tag, 0)
        return labels


# --------------------------------------------------------------
# 3. Metrics
# --------------------------------------------------------------
def compute_metrics(eval_pred):
    preds, labs = eval_pred
    preds = np.argmax(preds, axis=2)

    true_labels, pred_labels = [], []
    for i in range(labs.shape[0]):
        tl, pl = [], []
        for j in range(labs.shape[1]):
            if labs[i, j] != -100:
                tl.append(LABEL_MAP[labs[i, j]])
                pl.append(LABEL_MAP[preds[i, j]])
        true_labels.append(tl)
        pred_labels.append(pl)

    seqeval = evaluate.load("seqeval")
    res = seqeval.compute(
        predictions=pred_labels,
        references=true_labels,
        mode="strict",
        scheme="IOB2",
    )
    return {
        "precision": res["overall_precision"],
        "recall": res["overall_recall"],
        "f1": res["overall_f1"],
        "accuracy": res["overall_accuracy"],
    }


# --------------------------------------------------------------
# 4. Analysis helpers
# --------------------------------------------------------------
def analyze_dataset_entities(data, name="dataset"):
    counts = Counter()
    examples = {}
    for rec in data:
        for s, e, tag in rec["entities"]:
            etype = tag[2:] if tag.startswith(("B-", "I-")) else tag
            counts[etype] += 1
            examples.setdefault(etype, []).append(rec["text"][s:e])

    print("\n" + "=" * 50)
    print(f"Entity Distribution in {name}")
    print("=" * 50)
    print(f"Total samples : {len(data)}")
    print(f"Total entities: {sum(counts.values())}")
    for k, v in counts.most_common():
        ex = ", ".join(examples[k][:3])
        print(f"{k:>5}: {v:<5}   Examples: {ex}")
    return counts


def plot_entity_comparison(src_counts, tgt_counts):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.bar(src_counts.keys(), src_counts.values())
    ax1.set_title("Hiligaynon Entity Distribution")
    ax1.set_xlabel("Entity")
    ax1.set_ylabel("Count")

    ax2.bar(tgt_counts.keys(), tgt_counts.values())
    ax2.set_title("Tagalog Entity Distribution")
    ax2.set_xlabel("Entity")
    ax2.set_ylabel("Count")
    plt.tight_layout()
    plt.show()


# --------------------------------------------------------------
# 5. Evaluation pipeline
# --------------------------------------------------------------
def evaluate_crosslingual_transfer(
    model_path: str,
    hil_path: str,
    tgl_path: str,
    model_type: str = "xlm-roberta",
):
    global LABEL_MAP

    print("\n" + "=" * 60)
    print("Zero-shot Cross-lingual Transfer Evaluation")
    print(f"Model               : {model_type}")
    print("Source language     : Hiligaynon")
    print("Target language     : Tagalog")
    print("=" * 60)

    # ► Load data
    hil_data = load_hiligaynon_data(hil_path)
    tgl_data = load_tagalog_data(tgl_path)

    # ► Quick stats
    src_cnt = analyze_dataset_entities(hil_data, "Hiligaynon")
    tgt_cnt = analyze_dataset_entities(tgl_data, "Tagalog")
    plot_entity_comparison(src_cnt, tgt_cnt)

    # ► Model / tokenizer
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nLoading model from {model_path} on {device} ...")
    tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
    model = AutoModelForTokenClassification.from_pretrained(model_path).to(device)

    # ► Build datasets
    split = int(0.8 * len(hil_data))
    hil_test = hil_data[split:]

    hil_ds = HiligaynonNERDataset(
        [r["text"] for r in hil_test],
        [r["entities"] for r in hil_test],
        tokenizer,
    )
    tgl_ds = HiligaynonNERDataset(
        [r["text"] for r in tgl_data],
        [r["entities"] for r in tgl_data],
        tokenizer,
    )

    LABEL_MAP = tgl_ds.id_to_label  # for compute_metrics

    data_collator = DataCollatorForTokenClassification(
        tokenizer=tokenizer, pad_to_multiple_of=8, padding=True, label_pad_token_id=-100
    )

    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir="./eval_results",
            per_device_eval_batch_size=16,
            report_to="none",
            remove_unused_columns=False,
        ),
        data_collator=data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    # ► Evaluate
    print("\nRunning evaluation …")
    hil_res = trainer.evaluate(eval_dataset=hil_ds)
    tgl_res = trainer.evaluate(eval_dataset=tgl_ds)

    # ► Summarise
    print("\n" + "=" * 60)
    print("EVALUATION RESULTS")
    print("=" * 60)
    df = pd.DataFrame(
        {
            "Hiligaynon (in-language)": {
                "Precision": hil_res["eval_precision"],
                "Recall": hil_res["eval_recall"],
                "F1-Score": hil_res["eval_f1"],
                "Accuracy": hil_res["eval_accuracy"],
            },
            "Tagalog (zero-shot)": {
                "Precision": tgl_res["eval_precision"],
                "Recall": tgl_res["eval_recall"],
                "F1-Score": tgl_res["eval_f1"],
                "Accuracy": tgl_res["eval_accuracy"],
            },
        }
    )
    print(tabulate(df.T, headers="keys", tablefmt="github", floatfmt=".4f"))

    # ► Performance drop
    print("\n" + "=" * 60)
    print("CROSS-LINGUAL TRANSFER ANALYSIS")
    print("=" * 60)
    for metric in ["precision", "recall", "f1"]:
        drop = (
            (hil_res[f"eval_{metric}"] - tgl_res[f"eval_{metric}"])
            / hil_res[f"eval_{metric}"]
            * 100
        )
        print(f"{metric.capitalize()} drop: {drop:.2f}%")

    # ► Bar plot
    label_key_pairs = [
        ("Precision", "precision"),
        ("Recall",    "recall"),
        ("F1-Score",  "f1"),        # << no hyphen in the key
        ("Accuracy",  "accuracy"),
    ]

    labels      = [lbl for lbl, _ in label_key_pairs]
    hil_scores  = [hil_res[f"eval_{k}"] for _, k in label_key_pairs]
    tgl_scores  = [tgl_res[f"eval_{k}"] for _, k in label_key_pairs]


    x = np.arange(len(labels))
    width = 0.35
    fig, ax = plt.subplots(figsize=(10, 6))
    rects1 = ax.bar(x - width / 2, hil_scores, width, label="Hiligaynon")
    rects2 = ax.bar(x + width / 2, tgl_scores, width, label="Tagalog")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.1)
    ax.set_title("Cross-lingual Transfer: Hiligaynon → Tagalog")
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()

    for rect in rects1 + rects2:
        h = rect.get_height()
        ax.annotate(f"{h:.3f}", xy=(rect.get_x() + rect.get_width() / 2, h), xytext=(0, 3),
                    textcoords="offset points", ha="center", va="bottom")

    plt.tight_layout()
    plt.show()

    return df, hil_res, tgl_res


# --------------------------------------------------------------
# 6. Optional: small error analysis
# --------------------------------------------------------------
def perform_error_analysis(model, tokenizer, data, n=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    label_map = model.config.id2label

    print("\n" + "=" * 60)
    print(f"ERROR ANALYSIS (first {n} samples)")
    print("=" * 60)

    for i, rec in enumerate(data[:n]):
        text, gold = rec["text"], rec["entities"]
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True,
                           return_offsets_mapping=True)
        offsets = inputs.pop("offset_mapping")[0]
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            out = model(**inputs)
            ids = out.logits.argmax(-1)[0]

        # gather predicted spans
        preds = []
        cur = None
        for tok_id, (s, e) in zip(ids, offsets):
            if s == e:
                continue
            tag = label_map[tok_id.item()]
            if tag.startswith("B-"):
                if cur:
                    preds.append(cur)
                cur = {"text": text[s:e], "label": tag, "start": s, "end": e}
            elif tag.startswith("I-") and cur:
                cur["text"] = text[cur["start"]:e]
                cur["end"] = e
            else:
                if cur:
                    preds.append(cur)
                    cur = None
        if cur:
            preds.append(cur)

        print(f"\nSample {i+1}")
        print("Text :", text)
        print("Gold :", gold)
        print("Pred :", [(p["text"], p["label"]) for p in preds])


# --------------------------------------------------------------
# 7. Run
# --------------------------------------------------------------
if __name__ == "__main__":
    MODEL_PATH = "hiligaynon-ner-xlm-roberta-deploy"
    HIL_DATA = "final_combined_all.jsonl"
    TGL_DATA = "processed_tagalog_dataset_converted.jsonl"

    df, hil_res, tgl_res = evaluate_crosslingual_transfer(
        model_path=MODEL_PATH,
        hil_path=HIL_DATA,
        tgl_path=TGL_DATA,
        model_type="xlm-roberta",
    )

    # Optional error inspection
    tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
    model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH).to(
        torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )
    tagalog_data = load_tagalog_data(TGL_DATA)
    perform_error_analysis(model, tokenizer, tagalog_data, n=5)


## Results and analysis

 helpers (define once)

In [ ]:
# ======  helper functions & loaders  ======================================
import numpy as np, torch, evaluate, json
from seqeval.metrics import classification_report as seqeval_classification_report
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          Trainer, TrainingArguments,
                          DataCollatorForTokenClassification)

# ── (re)loader for converted Cebuano JSONL ────────────────────────────────
def load_cebuano_data(path):
    data = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            rec = json.loads(line.strip())
            ent = (rec['entities']
                   if 'entities' in rec else
                   [[lbl['start'], lbl['end'], lbl['labels'][0]] for lbl in rec['label']])
            data.append({'text': rec['text'], 'entities': ent})
    return data

# ── helper to run one cross-lingual evaluation & detailed report ──────────
def run_xling_eval(model_path, base_tok, src_jsonl, tgt_jsonl,
                   tgt_loader, tag_name):
    """Returns (overall_metrics_dict, seqeval_cls_report_str)."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # load data
    src_data = load_hiligaynon_data(src_jsonl)
    tgt_data = tgt_loader(tgt_jsonl)

    # tokenizer / model
    tokenizer = AutoTokenizer.from_pretrained(base_tok)
    model     = AutoModelForTokenClassification.from_pretrained(model_path).to(device)

    # dataset
    ds = HiligaynonNERDataset([r['text'] for r in tgt_data],
                              [r['entities'] for r in tgt_data],
                              tokenizer)

    data_collator = DataCollatorForTokenClassification(
        tokenizer=tokenizer, pad_to_multiple_of=8, padding=True, label_pad_token_id=-100)

    args = TrainingArguments(output_dir='./tmp', report_to='none',
                             per_device_eval_batch_size=16, remove_unused_columns=False)

    trainer = Trainer(model=model, args=args, data_collator=data_collator,
                      tokenizer=tokenizer)

    # overall metrics (precision / recall / f1 / accuracy)
    overall = trainer.evaluate(eval_dataset=ds)

    # token-level predictions for per-tag report
    pred_out  = trainer.predict(test_dataset=ds)
    logits    = pred_out.predictions
    label_ids = pred_out.label_ids
    preds = np.argmax(logits, axis=2)

    id2lab = ds.id_to_label
    true_tags, pred_tags = [], []
    for p_row, l_row in zip(preds, label_ids):
        t, p = [], []
        for p_i, l_i in zip(p_row, l_row):
            if l_i == -100:        # ignore padding
                continue
            t.append(id2lab[l_i]); p.append(id2lab[p_i])
        true_tags.append(t); pred_tags.append(p)

    cls_rep = seqeval_classification_report(true_tags, pred_tags, digits=3)

    print(f"\n{'='*66}\n{tag_name} — detailed tag report\n{'='*66}")
    print(cls_rep)
    return overall, cls_rep


Hiligaynon → Tagalog (mBERT vs XLM-R)

In [ ]:
# =================================================================================
# TOKEN-LEVEL IOB REPORTS (B-PER, I-ORG, …) + ONE WIDE SUMMARY TABLE
# ---------------------------------------------------------------------------------
# Assumptions:
#   • load_tagalog_data, load_cebuano_data, HiligaynonNERDataset,
#     AutoTokenizer, AutoModelForTokenClassification, Trainer,
#     TrainingArguments, DataCollatorForTokenClassification are
#     already defined in the notebook (from previous cells).
#   • Four fine-tuned models:
#         hiligaynon-ner-mbert-deploy
#         hiligaynon-ner-xlm-roberta-deploy
#   • Three JSONL datasets:
#         final_combined_all.jsonl              (Hiligaynon — not needed here)
#         processed_tagalog_dataset_converted.jsonl
#         cebuano_converted.jsonl
#
# This cell:
#   1) Defines token_level_report() → returns per-tag DataFrame
#   2) Runs the four model × language combinations
#   3) Merges the four DataFrames into one wide summary table
# =================================================================================

from sklearn.metrics import classification_report as sk_cls_report
from transformers import (AutoTokenizer, AutoModelForTokenClassification, Trainer,
                          TrainingArguments, DataCollatorForTokenClassification)
from tabulate import tabulate
import pandas as pd, numpy as np, torch

# ------------------------------------------------------------------------------
def token_level_report(model_path, base_tok, tgt_jsonl, tgt_loader, title):
    """Return DataFrame with token-level Precision / Recall / F1 / Support."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained(base_tok)
    model     = AutoModelForTokenClassification.from_pretrained(model_path).to(device)

    data = tgt_loader(tgt_jsonl)
    ds = HiligaynonNERDataset([r['text'] for r in data],
                              [r['entities'] for r in data],
                              tokenizer)

    trainer = Trainer(
        model=model,
        args=TrainingArguments(output_dir='./tmp', report_to='none',
                               per_device_eval_batch_size=16,
                               remove_unused_columns=False),
        data_collator=DataCollatorForTokenClassification(
            tokenizer, pad_to_multiple_of=8, padding=True, label_pad_token_id=-100))

    pred = trainer.predict(ds)
    preds  = np.argmax(pred.predictions, axis=2)
    label_ids = pred.label_ids
    id2lab = ds.id_to_label

    y_true, y_pred = [], []
    for p_row, l_row in zip(preds, label_ids):
        for p_i, l_i in zip(p_row, l_row):
            if l_i == -100:           # padding token
                continue
            y_true.append(id2lab[l_i])
            y_pred.append(id2lab[p_i])

    rpt = sk_cls_report(y_true, y_pred, output_dict=True, zero_division=0)
    df = (pd.DataFrame(rpt)
            .T
            .drop(index=['accuracy', 'macro avg', 'weighted avg', 'O'])
            .reset_index()
            .rename(columns={'index':'Tag',
                             'precision':'Precision',
                             'recall':'Recall',
                             'f1-score':'F1',
                             'support':'Support'}))
    print(f"\n{title}")
    print(tabulate(df, headers='keys', tablefmt='simple', floatfmt='.3f'))
    return df

# ------------------------------------------------------------------------------
# CONFIG
HIL_DATA = "final_combined_all.jsonl"                      # not used below
TGL_DATA = "processed_tagalog_dataset_converted.jsonl"
CEB_DATA = "cebuano_converted.jsonl"

MBERT = "hiligaynon-ner-mbert-deploy"
XLMR  = "hiligaynon-ner-xlm-roberta-deploy"

# ------------------------------------------------------------------------------
# RUN 4 evaluations and capture DataFrames
tgl_mbert_tags = token_level_report(
    MBERT, "bert-base-multilingual-cased",
    TGL_DATA, load_tagalog_data,
    "Tagalog  mBERT — per-tag metrics")

tgl_xlmr_tags = token_level_report(
    XLMR, "xlm-roberta-base",
    TGL_DATA, load_tagalog_data,
    "Tagalog  XLM-R — per-tag metrics")

ceb_mbert_tags = token_level_report(
    MBERT, "bert-base-multilingual-cased",
    CEB_DATA, load_cebuano_data,
    "Cebuano mBERT — per-tag metrics")

ceb_xlmr_tags = token_level_report(
    XLMR, "xlm-roberta-base",
    CEB_DATA, load_cebuano_data,
    "Cebuano XLM-R — per-tag metrics")

# ------------------------------------------------------------------------------
# MERGE INTO ONE WIDE SUMMARY TABLE
def tidy(df):
    # ensure 'Tag' column exists and is index
    if 'Tag' not in df.columns:
        df = df.reset_index().rename(columns={'index':'Tag'})
    return df.set_index('Tag')[['Precision','Recall','F1']]

reports = [
    (tidy(tgl_mbert_tags), "TGL-mBERT"),
    (tidy(tgl_xlmr_tags),  "TGL-XLMR"),
    (tidy(ceb_mbert_tags), "CEB-mBERT"),
    (tidy(ceb_xlmr_tags),  "CEB-XLMR"),
]

merged = None
for df, label in reports:
    merged = (df.add_prefix(f"{label}_") if merged is None
              else merged.join(df.add_prefix(f"{label}_"), how='outer'))

merged = merged.reset_index()
print("\nCombined per-tag metrics across all runs")
print(tabulate(merged, headers='keys', tablefmt='github', floatfmt='.3f'))


Summary